# SAAM 2026 — EUR Equity, Carbon-Aware Portfolio Allocation

**Region:** Europe &nbsp;•&nbsp; **Carbon scope:** 1 + 2 &nbsp;•&nbsp; **OOS window:** Jan 2014 – Dec 2025

This notebook implements the full SAAM pipeline end-to-end:

1. **Part I** — unconstrained minimum-variance allocation vs value-weighted benchmark.
2. **Part II §3.1** — baseline carbon profile of MV vs VW.
3. **Part II §3.2** — active investor: MV with $\mathrm{CF} \le 0.5 \cdot \mathrm{CF}(P^{(mv)})$.
4. **Part II §3.3** — passive investor: TE-min vs VW with $\mathrm{CF} \le 0.5 \cdot \mathrm{CF}(P^{(vw)})$.
5. **Part II §4.1** — net-zero glide path: TE-min vs VW with $\mathrm{CF} \le (1-\theta)^{Y-2012} \cdot \mathrm{CF}(P^{(vw)})_{2013}$, $\theta = 10\%$.

> **Headline findings.** Decarbonizing the active investor's MV portfolio by 50% costs essentially zero (Sharpe drops by 0.001). Decarbonizing the passive VW portfolio by 50% costs ~30 bp/year of return at 80 bp of TE (IR = −0.41), statistically indistinguishable from zero given 144 monthly observations. A 10% / year net-zero glide path was *looser* than a "CF ≤ 50% of benchmark" rule in every year of our window because the benchmark itself decarbonized by 58% organically.

## 0 — Setup

Imports, configuration, and a shared visual palette. All plots in this notebook use a single colour scheme so the eye learns each portfolio:

| Portfolio | Colour | Role |
|---|---|---|
| `VW` | charcoal | benchmark |
| `MV` | navy | active baseline |
| `MV(0.5)` | terracotta | active decarbonized |
| `VW(0.5)` | teal | passive decarbonized (50% rule) |
| `VW(NZ)` | purple | passive decarbonized (10%/yr glide) |
| Targets / glides | dotted grey | constraint lines |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from scipy.optimize import minimize, linprog
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XlImage
from IPython.display import display, Markdown

# ---------------------------------------------------------------------------
# Configuration — edit paths if your data lives elsewhere
# ---------------------------------------------------------------------------
REGION          = "EUR"
START_YEAR      = 2013
END_YEAR        = 2024
ESTIM_YEARS     = 10
MIN_OBS         = 36
STALE_THR       = 0.30
LOW_FLOOR       = 0.50
LW_SHRINK_FLOOR = 0.01

DATA_PATH      = "Data_2026/"
TEMPLATE_PATH  = "Data_2026/Template_for_Part_I-SAAM.xlsx"
OUT            = "Output_2026/"

import os
os.makedirs(OUT, exist_ok=True)

# ---------------------------------------------------------------------------
# Plot styling — single coherent palette, applied globally
# ---------------------------------------------------------------------------
PAL = {
    "vw":     "#2C2C2C",   # near-black benchmark
    "mv":     "#1F4E79",   # navy active baseline
    "mv05":   "#E76F51",   # terracotta MV(0.5)
    "vw05":   "#2A9D8F",   # teal VW(0.5)
    "vwnz":   "#9B59B6",   # purple VW(NZ)
    "target": "#888888",   # neutral grey for targets / glides
    "event":  "#F4A261",   # warm peach for event shading
    "high_e": "#D62828",   # high-emission accent
    "low_e":  "#2D6A4F",   # low-emission accent
    "grid":   "#E5E5E5",
}

plt.rcParams.update({
    "figure.figsize":      (12, 5),
    "figure.dpi":          110,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.titlepad":       10,
    "axes.labelsize":      10,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.color":          PAL["grid"],
    "grid.linewidth":      0.6,
    "grid.alpha":          0.7,
    "legend.frameon":      False,
    "legend.fontsize":     9,
    "xtick.labelsize":     9,
    "ytick.labelsize":     9,
    "font.family":         "DejaVu Sans",
    "font.size":           10,
})

# ---------------------------------------------------------------------------
# Annotation helpers — used across all plots for consistent storytelling
# ---------------------------------------------------------------------------
COVID_START   = pd.Timestamp("2020-02-01")
COVID_END     = pd.Timestamp("2020-04-30")
ENERGY_START  = pd.Timestamp("2021-06-01")
ENERGY_END    = pd.Timestamp("2022-12-31")

def shade_events(ax, covid=True, energy=True, label_first=True):
    """Light shading on COVID crash and 2021-22 energy bull market.
    These are the two regimes the findings doc attributes most active-return
    drift to. label_first=True puts text labels at the top of the first axes."""
    if covid:
        ax.axvspan(COVID_START, COVID_END, color=PAL["event"], alpha=0.18, lw=0)
    if energy:
        ax.axvspan(ENERGY_START, ENERGY_END, color=PAL["event"], alpha=0.10, lw=0)
    if label_first:
        y_top = ax.get_ylim()[1]
        ax.text(COVID_START + (COVID_END - COVID_START)/2, y_top,
                " COVID ", ha="center", va="top", fontsize=8,
                color="#7A4017", style="italic")
        ax.text(ENERGY_START + (ENERGY_END - ENERGY_START)/2, y_top,
                " Energy bull ", ha="center", va="top", fontsize=8,
                color="#7A4017", style="italic")

def label_endpoint(ax, x, y, txt, color, dx=8, dy=0, fontsize=9, weight="bold"):
    ax.annotate(txt, xy=(x, y), xytext=(dx, dy), textcoords="offset points",
                color=color, fontsize=fontsize, fontweight=weight, va="center")

def year_axis(ax, every=2):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(every))

# Plain percent formatter for axes (not ticks at integer values)
PCT0 = FuncFormatter(lambda y, _: f"{y:.0f}%")
PCT1 = FuncFormatter(lambda y, _: f"{y:.1f}%")

---
## Part I — Standard Allocation: Min-Variance vs Value-Weighted

### 1. Load raw Datastream files

The `Static_2025.xlsx` file gives the EUR firm list (region assignment, country, ISIN, name). All other files are panel data keyed on ISIN: monthly total-return index `RI`, monthly market cap `MV`, annual scope-1 and scope-2 emissions, annual revenue, plus the Fama–French risk-free rate.

In [ ]:
print("=" * 65)
print("SAAM Part I — Minimum Variance Portfolio (EUR region)")
print("=" * 65)
print("\n[1] Loading raw data ...")

static     = pd.read_excel(DATA_PATH + "Static_2025.xlsx")
ri_m_raw   = pd.read_excel(DATA_PATH + "DS_RI_T_USD_M_2025.xlsx",   sheet_name="RI")
mv_m_raw   = pd.read_excel(DATA_PATH + "DS_MV_T_USD_M_2025.xlsx",   sheet_name="MV")
ri_y_raw   = pd.read_excel(DATA_PATH + "DS_RI_T_USD_Y_2025.xlsx",   sheet_name="RI")
mv_y_raw   = pd.read_excel(DATA_PATH + "DS_MV_T_USD_Y_2025.xlsx",   sheet_name="MV")
co2_s1_raw = pd.read_excel(DATA_PATH + "DS_CO2_SCOPE_1_Y_2025.xlsx", sheet_name="Scope1")
co2_s2_raw = pd.read_excel(DATA_PATH + "DS_CO2_SCOPE_2_Y_2025.xlsx", sheet_name="Scope2")
rev_raw    = pd.read_excel(DATA_PATH + "DS_REV_Y_2025.xlsx",         sheet_name="REV")
rf_raw     = pd.read_excel(DATA_PATH + "Risk_Free_Rate_2025.xlsx",
                           sheet_name="F-F_Research_Data_Factors")

print(f"   Static: {static.shape}, RI monthly: {ri_m_raw.shape}")

### 2. Clean the Datastream wide format

Datastream exports panels with ISIN as a row identifier and dates as columns. The first task is mechanical:

- Drop rows with no ISIN or where the ISIN column contains an `ER:` error tag.
- Restrict to ISINs in our EUR static set.
- Coerce data columns to numeric and convert date headers to `pd.Timestamp` where applicable.

This same `clean_ds()` function is applied uniformly to all eight raw frames.

In [ ]:
print("\n[2] Cleaning Datastream format ...")

def clean_ds(df, valid_isins):
    """Drop error rows, filter to valid ISINs, set ISIN as index, coerce numeric."""
    df = df.dropna(subset=["ISIN"]).copy()
    df = df[~df["ISIN"].astype(str).str.contains("ER:", na=False)]
    df = df[df["ISIN"].isin(valid_isins)]
    names = df.set_index("ISIN")["NAME"]
    data_cols = [c for c in df.columns if c not in ["NAME", "ISIN"] and c is not None]
    data = df.set_index("ISIN")[data_cols].apply(pd.to_numeric, errors="coerce")
    new_cols = []
    for c in data.columns:
        if hasattr(c, "year") and not isinstance(c, (int, np.integer)):
            new_cols.append(pd.Timestamp(c))
        else:
            new_cols.append(c)
    data.columns = new_cols
    return data, names

eur_isins = set(static.loc[static["Region"] == REGION, "ISIN"])
print(f"   EUR ISINs in static: {len(eur_isins)}")

ri_m,   firm_names = clean_ds(ri_m_raw,   eur_isins)
mv_m,   _          = clean_ds(mv_m_raw,   eur_isins)
ri_y,   _          = clean_ds(ri_y_raw,   eur_isins)
mv_y,   _          = clean_ds(mv_y_raw,   eur_isins)
co2_s1, _          = clean_ds(co2_s1_raw, eur_isins)
co2_s2, _          = clean_ds(co2_s2_raw, eur_isins)
rev,    _          = clean_ds(rev_raw,    eur_isins)

isin_name = firm_names.to_dict()
print(f"   EUR firms loaded: {ri_m.shape[0]}")

### 3. Build date lists (precomputed for O(1) lookup)

`monthly_all` and `annual_all` give the union of available dates across all monthly / annual panels. We pre-build:

- `_months_of[Y]` — list of 12 month-ends in year `Y`,
- `_estim_window[Y]` — list of 120 month-ends in `Y−9 … Y` (the 10-year rolling window for estimation),
- `_monthly_idx[t]` — position of `t` in `monthly_all`, for fast neighbour lookup.

The Dec-2013 estimation window must contain exactly 120 months — verified by assertion below.

In [ ]:
monthly_all = sorted([c for c in ri_m.columns
                      if isinstance(c, pd.Timestamp)
                      and pd.Timestamp("2000-01-01") <= c <= pd.Timestamp("2025-12-31")])
annual_all  = sorted([c for c in ri_y.columns if isinstance(c, (int, np.integer))])

ri_m   = ri_m[monthly_all]
mv_m   = mv_m[[c for c in monthly_all if c in mv_m.columns]]
ri_y   = ri_y[[c for c in annual_all  if c in ri_y.columns]]
mv_y   = mv_y[[c for c in annual_all  if c in mv_y.columns]]
co2_s1 = co2_s1[[c for c in annual_all if c in co2_s1.columns]]
co2_s2 = co2_s2[[c for c in annual_all if c in co2_s2.columns]]
rev    = rev[[c for c in annual_all   if c in rev.columns]]

_monthly_idx = {t: i for i, t in enumerate(monthly_all)}

def months_of(Y):
    return [c for c in monthly_all if c.year == Y]

def estim_window(Y):
    return [c for c in monthly_all
            if (Y - ESTIM_YEARS + 1, 1) <= (c.year, c.month) <= (Y, 12)]

_months_of    = {Y: months_of(Y)    for Y in range(START_YEAR - 1, END_YEAR + 2)}
_estim_window = {Y: estim_window(Y) for Y in range(START_YEAR, END_YEAR + 1)}

ew2013 = _estim_window[2013]
print(f"   Estimation window Dec 2013: {len(ew2013)} months "
      f"({ew2013[0].strftime('%Y-%m')} to {ew2013[-1].strftime('%Y-%m')})")
assert len(ew2013) == 120, f"Expected 120, got {len(ew2013)}"

### 4. Risk-free rate

The Fama–French file stores the RF column as **monthly returns in percent**, per instructor directive — so we divide by 100 to get a monthly decimal. To annualize the average for the Sharpe denominator, we multiply by 12 (matching the convention used for $\mu_{\text{arith,ann}} = 12 \cdot \bar R_m$). A runtime assertion verifies that the annualized average sits in a plausible 0.5%–5% band; over our 2014–2025 window it lands at ~1.75%, consistent with the average US 3-month T-bill yield.

In [ ]:
print("\n[3] Processing risk-free rate ...")

rf_raw.columns = ["YYYYMM", "RF_pct"]
rf_raw = rf_raw.dropna(subset=["YYYYMM"])
rf_raw["date"] = pd.to_datetime(rf_raw["YYYYMM"].astype(int).astype(str), format="%Y%m")
rf_raw["date"] = rf_raw["date"] + pd.offsets.MonthEnd(0)
rf_mon = rf_raw.set_index("date")["RF_pct"] / 100
rf_mon = rf_mon.squeeze()
rf_mon.name = "RF"

print(f"   RF range: {rf_mon.index.min().strftime('%Y-%m')} to "
      f"{rf_mon.index.max().strftime('%Y-%m')}")
print(f"   RF sample 2014-01: {rf_mon.loc['2014-01'].values[0]:.6f} (monthly)")

_rf_ann_check = rf_mon.loc["2014-01-01":"2025-12-31"].mean() * 12
assert 0.005 < _rf_ann_check < 0.05, \
    f"Annualised avg rf = {_rf_ann_check:.4%} — outside plausible range."
print(f"   Sanity: annualised avg rf over 2014-2025 = {_rf_ann_check*100:.4f}%")

### 5. Price cleaning — three steps

**Low-price floor.** Datastream's RI is occasionally rounded to near-zero, causing spurious 1000%+ returns when prices recover. We treat any RI below 0.50 as missing.

**Forward-fill internal gaps.** Missing RI values *between* a firm's first and last valid observation are forward-filled. This avoids look-ahead bias (only past info is used) and is consistent with the project instruction. Leading NaN (firm not yet listed) and trailing NaN (delisted) are deliberately left alone.

**Delisting handling — most punitive treatment.** Whenever a firm's last valid price falls before 2025-12-31, we apply a −100% return in the next month and NaN thereafter. This eliminates survivorship bias in OOS returns at the cost of being conservative about acquisitions-at-premium. The same logic is reapplied during OOS portfolio simulation via `fill_oos_returns()`.

> *Source: project instructions §1; Shumway (1997), "The Delisting Bias in CRSP Data".*

In [ ]:
print("\n[4] Cleaning prices ...")

n_low = ((ri_m > 0) & (ri_m < LOW_FLOOR)).sum().sum()
ri_m[ri_m < LOW_FLOOR] = np.nan
print(f"   Prices < {LOW_FLOOR} set to NaN: {n_low}")

def forward_fill_internal(row):
    fv = row.first_valid_index()
    lv = row.last_valid_index()
    if fv is None or lv is None:
        return row
    cols = list(row.index)
    start = cols.index(fv)
    end   = cols.index(lv)
    row.iloc[start:end + 1] = row.iloc[start:end + 1].ffill()
    return row

ri_before = ri_m.isna().sum().sum()
ri_m = ri_m.apply(forward_fill_internal, axis=1)
n_filled = ri_before - ri_m.isna().sum().sum()
print(f"   Forward-filled internal gaps: {n_filled} observations")

cutoff = pd.Timestamp("2025-12-31")
last_valid = {}
for isin in ri_m.index:
    lv = ri_m.loc[isin].last_valid_index()
    if lv is not None and isinstance(lv, pd.Timestamp) and lv < cutoff:
        last_valid[isin] = lv
print(f"   Firms with early last price (potential delistings): {len(last_valid)}")

ret_m = ri_m.pct_change(axis=1)

for isin, ddate in last_valid.items():
    if ddate not in _monthly_idx:
        continue
    idx = _monthly_idx[ddate]
    if idx + 1 < len(monthly_all):
        ret_m.at[isin, monthly_all[idx + 1]] = -1.0
        for k in range(idx + 2, len(monthly_all)):
            ret_m.at[isin, monthly_all[k]] = np.nan

ret_m = ret_m.iloc[:, 1:]

n_delist = (ret_m == -1.0).sum().sum()
print(f"   -100% delisting returns applied: {n_delist}")

# Annual carbon and revenue: forward-fill along the time axis
co2_s1 = co2_s1.ffill(axis=1)
co2_s2 = co2_s2.ffill(axis=1)
rev    = rev.ffill(axis=1)

### 6. Investment-set construction

At end of each year `Y`, the universe for `Y+1` is built by four sequential filters — all using only information available up to Dec `Y`:

1. **Valid Dec-`Y` price** in monthly RI (after low-floor + forward-fill).
2. **At least 36 valid monthly returns** in the 120-month estimation window ending Dec `Y`.
3. **No more than 30% zero-return months** in that same window (Lesmond, Ogden & Trzcinka 1999 stale-price proxy; the project instructions suggested 50% but the teacher tightened to 30%).
4. **Both Scope 1 and Scope 2 emissions present** at end of `Y`.

Filter 4 is required *even in Part I* — the project specification asks that the same investment set be used for both parts so portfolios are directly comparable.

In [ ]:
print("\n[5] Building investment sets ...")

def get_universe(Y):
    """Build the year-end-Y eligibility set for allocation in Y+1."""
    dec_cols = [c for c in monthly_all if c.year == Y and c.month == 12]
    if not dec_cols:
        return []
    dec_Y = dec_cols[0]

    win = _estim_window[Y]
    win_ret = [c for c in win if c in ret_m.columns]
    R_win = ret_m.reindex(columns=win_ret)

    out = []
    for isin in ri_m.index:
        if pd.isna(ri_m.at[isin, dec_Y]):
            continue
        if isin not in R_win.index:
            continue
        row = R_win.loc[isin]
        n_valid = row.notna().sum()
        if n_valid < MIN_OBS:
            continue
        n_zero = ((row == 0) | (row.abs() < 1e-10)).sum()
        if (n_zero / n_valid) > STALE_THR:
            continue
        has_s1 = Y in co2_s1.columns and pd.notna(co2_s1.at[isin, Y]) if isin in co2_s1.index else False
        has_s2 = Y in co2_s2.columns and pd.notna(co2_s2.at[isin, Y]) if isin in co2_s2.index else False
        if not (has_s1 and has_s2):
            continue
        out.append(isin)
    return out

universe = {}
for Y in range(START_YEAR, END_YEAR + 1):
    universe[Y] = get_universe(Y)

years_part2 = list(range(START_YEAR, END_YEAR + 1))

uni_size = pd.Series({Y: len(universe[Y]) for Y in years_part2}, name="N_firms")
uni_size.index.name = "Year"
display(uni_size.to_frame().T)

### 7. Covariance estimation — pairwise complete + Ledoit-Wolf shrinkage (OAS)

**Stage 1 — pairwise complete sample covariance.** For each firm $i$ we compute the variance over its full available sample. For each pair $(i,j)$ we compute the correlation over the months where *both* are observed, then reconstruct $\mathrm{Cov}(i,j) = \rho_{ij} \sqrt{\mathrm{Var}(i)\,\mathrm{Var}(j)}$. Maximises data usage but isn't guaranteed PSD.

**Stage 2 — shrinkage to a constant-correlation target.** $\Sigma_{\text{shrunk}} = \delta F + (1-\delta) S$, where $F$ keeps each firm's own variance but replaces all correlations with the average off-diagonal correlation. The intensity $\delta$ uses the **OAS closed-form approximation** (Chen et al. 2010) — no cross-validation required. A floor of `LW_SHRINK_FLOOR = 0.01` prevents zero shrinkage; final spectral clipping at $10^{-10}$ guarantees PSD.

> *Sources: Lecture 5 slides 20–21; Stambaugh (1997); Ledoit & Wolf (2003, 2004); Chen et al. (2010).*

> ⚠ **Ex-ante TE caveat.** This shrinkage target *damps the carbon-sector factor*. The resulting $\Sigma$ assigns near-zero variance to a "long staples / short utilities-energy-materials" tilt — exactly the direction the carbon constraint pushes us. Realised TE will be ~16× ex-ante TE. We diagnose this explicitly in §10 of this notebook.

In [ ]:
_cov_cache = {}

def estimate_cov(isins, win_cols):
    """Pairwise-complete sample cov + OAS shrinkage to constant-correlation target."""
    win_in = [c for c in win_cols if c in ret_m.columns]
    R = ret_m.loc[isins, win_in]
    N = len(isins)

    R_vals = R.values
    not_nan = ~np.isnan(R_vals)
    mu_full = np.nanmean(R_vals, axis=1)
    R_demeaned = R_vals - mu_full[:, None]
    R_demeaned_zero = np.where(not_nan, R_demeaned, 0.0)
    n_obs = not_nan.sum(axis=1)
    var = np.where(n_obs > 1,
                   np.sum(R_demeaned_zero ** 2, axis=1) / n_obs, 0.0)

    R_zero = np.where(not_nan, R_vals, 0.0)
    not_nan_f = not_nan.astype(np.float64)

    count_ij = not_nan_f @ not_nan_f.T
    sum_i_ij = R_zero @ not_nan_f.T
    sum_j_ij = not_nan_f @ R_zero.T

    safe_count = np.maximum(count_ij, 1)
    mean_i_ij = sum_i_ij / safe_count
    mean_j_ij = sum_j_ij / safe_count

    cross_ij = R_zero @ R_zero.T
    cov_ij   = (cross_ij / safe_count) - mean_i_ij * mean_j_ij

    sum_sq_i_ij = (R_zero ** 2) @ not_nan_f.T
    var_i_ij    = np.maximum(sum_sq_i_ij / safe_count - mean_i_ij ** 2, 0)
    sum_sq_j_ij = not_nan_f @ (R_zero ** 2).T
    var_j_ij    = np.maximum(sum_sq_j_ij / safe_count - mean_j_ij ** 2, 0)

    denom = np.sqrt(var_i_ij * var_j_ij)
    corr_ij = np.where(denom > 1e-20, cov_ij / denom, 0.0)

    std_own = np.sqrt(var)
    Sig = corr_ij * np.outer(std_own, std_own)
    np.fill_diagonal(Sig, var)
    Sig = (Sig + Sig.T) / 2

    # OAS shrinkage to constant-correlation target
    std_diag = np.sqrt(np.diag(Sig))
    std_diag_safe = np.where(std_diag > 1e-20, std_diag, 1e-20)
    corr_mat = np.clip(Sig / np.outer(std_diag_safe, std_diag_safe), -1.0, 1.0)
    np.fill_diagonal(corr_mat, 1.0)
    rho_bar = (corr_mat.sum() - N) / (N * (N - 1))
    F = rho_bar * np.outer(std_diag, std_diag)
    np.fill_diagonal(F, var)

    tr_Sig  = np.trace(Sig)
    tr_Sig2 = np.trace(Sig @ Sig)
    T_eff = max(np.median(count_ij[np.triu_indices(N, k=1)]), 2)
    numerator   = (1.0 - 2.0 / N) * tr_Sig2 + tr_Sig ** 2
    denominator = (T_eff + 1.0 - 2.0 / N) * (tr_Sig2 - tr_Sig ** 2 / N)
    delta = 0.5 if abs(denominator) < 1e-20 else max(min(numerator / denominator, 1.0),
                                                     LW_SHRINK_FLOOR)

    Sig_shrunk = delta * F + (1.0 - delta) * Sig
    eigvals, eigvecs = np.linalg.eigh(Sig_shrunk)
    if eigvals[0] < 1e-10:
        eigvals = np.maximum(eigvals, 1e-10)
        Sig_shrunk = eigvecs @ np.diag(eigvals) @ eigvecs.T
        Sig_shrunk = (Sig_shrunk + Sig_shrunk.T) / 2

    mu = np.nanmean(R_vals, axis=1)
    return mu, Sig_shrunk

def get_cov(Y):
    if Y not in _cov_cache:
        _cov_cache[Y] = estimate_cov(universe[Y], _estim_window[Y])
    return _cov_cache[Y]

### 8. OOS-return cache and value-weighted helper

`fill_oos_returns(Y)` materialises the next-year monthly return matrix for `universe[Y]` with explicit delisting-aware logic: a missing return where the previous month had a valid price → −100%; missing-with-no-prior-price → 0%. Cached so subsequent sections (MV, MV(0.5), VW(0.5), VW(NZ)) reuse the same matrix.

`vw_weights(Y)` returns the year-end value weights for `universe[Y]`, also cached.

In [ ]:
def fill_oos_returns(eligible, next_months):
    R_oos = ret_m.loc[eligible].reindex(columns=next_months).copy()
    for isin in eligible:
        delisted = False
        for k, t in enumerate(next_months):
            if delisted:
                R_oos.at[isin, t] = np.nan
                continue
            if pd.isna(R_oos.at[isin, t]):
                t_idx = _monthly_idx.get(t, None)
                prev_t = monthly_all[t_idx - 1] if t_idx and t_idx > 0 else None
                had_price_prev = (prev_t is not None
                                  and prev_t in ri_m.columns
                                  and isin in ri_m.index
                                  and pd.notna(ri_m.at[isin, prev_t]))
                if had_price_prev:
                    R_oos.at[isin, t] = -1.0
                    delisted = True
                else:
                    R_oos.at[isin, t] = 0.0
    return R_oos.fillna(0.0)

_oos_cache = {}
def get_oos_returns(Y):
    if Y not in _oos_cache:
        _oos_cache[Y] = fill_oos_returns(universe[Y], _months_of[Y + 1])
    return _oos_cache[Y]

_vww_cache = {}
def vw_weights(Y):
    if Y not in _vww_cache:
        isins = universe[Y]
        cap = mv_y.loc[isins, Y].fillna(0.0)
        s = cap.sum()
        _vww_cache[Y] = (cap / s if s > 0
                         else pd.Series(np.ones(len(isins)) / len(isins), index=isins))
    return _vww_cache[Y]

### 9. Rolling minimum-variance optimisation

For each Dec `Y` we solve the long-only GMV problem:

$$
\min_\alpha \alpha' \Sigma_Y \alpha \quad \text{s.t.} \quad \alpha' \mathbf{e} = 1, \quad \alpha_i \ge 0 \,\,\forall i.
$$

**Solver:** SLSQP via `scipy.optimize.minimize` with analytical gradient $\nabla = 2\Sigma w$, `ftol = 1e-10`, `maxiter = 1000`.

**Warm start.** Instead of $1/N$, the optimiser is initialised from the *drifted* portfolio at the end of the previous year (after 12 months of price-driven weight drift). This mirrors what an actual rebalancing investor would do and tends to reduce iterations.

**No upper bound on individual weights.** The project specification only requires non-negativity. As Jagannathan & Ma (2003) show, the long-only constraint already acts as implicit shrinkage, mitigating extreme concentration. *Adding a 5% or 10% per-firm cap would be a useful robustness check.*

In [ ]:
print("\n[6] Rolling min-variance optimisation ...")

_drifted_w = {}

def min_var_weights(Sigma, isins, Y):
    N = Sigma.shape[0]
    if Y in _drifted_w:
        prev = _drifted_w[Y]
        w0 = np.array([prev.get(i, 0.0) for i in isins])
        w0 = w0 / w0.sum() if w0.sum() > 0 else np.ones(N) / N
    else:
        w0 = np.ones(N) / N

    res = minimize(
        fun=lambda w: float(w @ Sigma @ w),
        x0=w0,
        jac=lambda w: 2.0 * (Sigma @ w),
        method="SLSQP",
        bounds=[(0.0, None)] * N,
        constraints={"type": "eq", "fun": lambda w: w.sum() - 1.0},
        options={"ftol": 1e-10, "maxiter": 1000},
    )
    if not res.success:
        print(f"   WARNING: optimizer did not converge for Y={Y}: {res.message}")
    return res.x

mv_w_dict, mv_ret = {}, {}

for Y in range(START_YEAR, END_YEAR + 1):
    eligible = universe[Y]
    N = len(eligible)
    if N == 0:
        continue

    mu, Sig = get_cov(Y)
    w = min_var_weights(Sig, eligible, Y)
    mv_w_dict[Y] = pd.Series(w, index=eligible)

    next_months = _months_of[Y + 1]
    R_next = get_oos_returns(Y)
    ww = w.copy()
    port_ret = []
    for t in next_months:
        r_t = R_next[t].values
        rp_t = float(ww @ r_t)
        port_ret.append(rp_t)
        ww = ww * (1.0 + r_t) / max(1.0 + rp_t, 1e-12)
    mv_ret[Y + 1] = pd.Series(port_ret, index=next_months)
    _drifted_w[Y + 1] = dict(zip(eligible, ww))

rp_mv = pd.concat(mv_ret).droplevel(0).sort_index()
rp_mv.index = pd.DatetimeIndex(rp_mv.index)
print(f"   MV monthly OOS returns: {len(rp_mv)} months")

### 10. Value-weighted benchmark

Same investment set as MV. Each month $t$, weight $w_{i,t} = \mathrm{Cap}_{i,t-1}\,/\,\sum_j \mathrm{Cap}_{j,t-1}$. Firms with missing month-end cap receive zero weight for that month and may re-enter the next. The same delisting-aware return matrix (`fill_oos_returns`) is used for both MV and VW, so any return penalty from a delisting is applied identically to both.

In [ ]:
print("\n[7] Value-weighted benchmark ...")

vw_ret = {}
for Y in range(START_YEAR, END_YEAR + 1):
    eligible = universe[Y]
    next_months = _months_of[Y + 1]
    R_oos_vw = get_oos_returns(Y)
    port = []
    for t in next_months:
        idx = _monthly_idx.get(t)
        if idx is None or idx == 0:
            port.append(np.nan)
            continue
        prev_t = monthly_all[idx - 1]
        cap = (mv_m.loc[eligible, prev_t].fillna(0)
               if prev_t in mv_m.columns else pd.Series(0.0, index=eligible))
        tot = cap.sum()
        if tot <= 0:
            port.append(0.0)
            continue
        r_t = R_oos_vw[t].values if t in R_oos_vw.columns else np.zeros(len(eligible))
        port.append(float((cap / tot).values @ r_t))
    vw_ret[Y + 1] = pd.Series(port, index=next_months)

rp_vw = pd.concat(vw_ret).droplevel(0).sort_index()
rp_vw.index = pd.DatetimeIndex(rp_vw.index)
print(f"   VW monthly OOS returns: {len(rp_vw)} months")

### 11. Performance statistics + verification

Conventions per Lecture 5 / project formula:

- $\mu_{\text{arith}}^{\text{ann}} = 12 \cdot \overline{R_m}$
- $\mu_{\text{geom}}^{\text{ann}} = (1 + R_{\text{cum}})^{12/T} - 1$
- $\sigma^{\text{ann}} = \sigma_m \cdot \sqrt{12}$, with $\sigma_m$ using the unbiased $1/(T-1)$ estimator (vs $1/T$ for the cov matrix; this minor inconsistency follows each source formula in its own context)
- $\mathrm{SR}^{\text{ann}} = (\mu_{\text{arith}}^{\text{ann}} - r_f^{\text{ann}}) / \sigma^{\text{ann}}$
- Max DD = peak-to-trough on cumulative returns

In [ ]:
print("\n[8] Performance statistics ...")

def compute_perf(rp, rf_s, label):
    rp = rp.dropna()
    rf = rf_s.reindex(rp.index).ffill().fillna(0)
    T = len(rp)
    mu_arith = 12 * rp.mean()
    mu_geom  = (1 + rp).prod() ** (12 / T) - 1
    sig_ann  = rp.std() * np.sqrt(12)
    rf_ann   = 12 * rf.mean()
    SR = (mu_arith - rf_ann) / sig_ann
    cum = (1 + rp).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()
    return {
        "Portfolio": label,
        "Ann. Return Arith. (%)": round(mu_arith * 100, 2),
        "Ann. Return Geom. (%)":  round(mu_geom * 100, 2),
        "Ann. Vol (%)":           round(sig_ann * 100, 2),
        "Sharpe":                 round(SR, 3),
        "Min Mo. (%)":            round(rp.min() * 100, 2),
        "Max Mo. (%)":            round(rp.max() * 100, 2),
        "Max DD (%)":             round(mdd * 100, 2),
    }

stats_df = pd.DataFrame([
    compute_perf(rp_vw, rf_mon, "Val-Wgt P^(vw)"),
    compute_perf(rp_mv, rf_mon, "Min-Var P_oos^(mv)"),
]).set_index("Portfolio")

display(stats_df)

# Verification ---------------------------------------------------------------
for Y, w in mv_w_dict.items():
    assert abs(w.sum() - 1.0) < 1e-6, f"Y={Y}: weights sum to {w.sum()}"
    assert (w >= -1e-8).all(),         f"Y={Y}: negative weights found"
assert rp_mv.isna().sum() == 0, "NaN in min-var returns"
assert rp_vw.isna().sum() == 0, "NaN in VW returns"
print("✓ All weight vectors sum to 1, all weights non-negative, no NaN OOS returns.")

### 12. Top-10 MV holdings — concentration snapshots

In [ ]:
def top_holdings_table(Y, k=10):
    if Y not in mv_w_dict:
        return None
    w = mv_w_dict[Y].sort_values(ascending=False)
    rows = []
    for rk, (isin, wt) in enumerate(w.head(k).items(), 1):
        cty = static.loc[static["ISIN"] == isin, "Country"].values
        rows.append({"Rank": rk, "Firm": isin_name.get(isin, isin)[:40],
                     "Country": cty[0] if len(cty) else "",
                     "Weight (%)": round(wt * 100, 2)})
    df = pd.DataFrame(rows).set_index("Rank")
    n_nz = (w > 1e-6).sum()
    df.attrs["caption"] = f"Dec {Y} (non-zero positions: {n_nz}/{len(w)})"
    return df

for Y in [2013, 2018, 2024]:
    df = top_holdings_table(Y)
    print(f"\n— Top-10 MV holdings, Dec {Y}  (non-zero / N = "
          f"{(mv_w_dict[Y] > 1e-6).sum()}/{len(mv_w_dict[Y])}) —")
    display(df)

### 13. Part I figures — Min-Var vs Value-Weighted

Four panels:
1. **Cumulative return** (base = 1 at Jan 2014). COVID (Feb–Apr 2020) and the 2021–2022 energy bull market are shaded — both regimes matter for the carbon-tilted analysis later.
2. **Rolling 12-month annualized volatility.** The MV portfolio's whole reason for existing is the lower line here.
3. **Drawdown from peak.** MV trades return for drawdown stability.
4. **Universe size by year.** Coverage grows as carbon disclosure improves; the dip in 2024 reflects the Y-end carbon-data filter on the most recent year.

In [ ]:
print("\n[9] Part I figure ...")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Part I — EUR: Min-Var vs Value-Weighted (Jan 2014 – Dec 2025)",
             fontsize=14, fontweight="bold", y=1.00)

# ── Panel 1: cumulative return ────────────────────────────────────────────
ax = axes[0, 0]
cm = (1 + rp_mv.dropna()).cumprod()
cv = (1 + rp_vw.dropna()).cumprod()
ax.plot(cm.index, cm.values, color=PAL["mv"], lw=1.9, label=r"Min-Var $P_{oos}^{(mv)}$")
ax.plot(cv.index, cv.values, color=PAL["vw"], lw=1.9, ls="--", label=r"Val-Wgt $P^{(vw)}$")
ax.set_title("Cumulative return (base = 1 at Jan 2014)")
ax.set_ylabel("Wealth multiple")
ax.axhline(1, color="#BBBBBB", lw=0.6, zorder=0)
shade_events(ax, label_first=True)
label_endpoint(ax, cm.index[-1], cm.iloc[-1], f" {cm.iloc[-1]:.2f}×", PAL["mv"])
label_endpoint(ax, cv.index[-1], cv.iloc[-1], f" {cv.iloc[-1]:.2f}×", PAL["vw"])
ax.legend(loc="upper left")
year_axis(ax)

# ── Panel 2: rolling vol ──────────────────────────────────────────────────
ax = axes[0, 1]
rv_mv = rp_mv.rolling(12).std() * np.sqrt(12) * 100
rv_vw = rp_vw.rolling(12).std() * np.sqrt(12) * 100
ax.plot(rv_mv.index, rv_mv.values, color=PAL["mv"], lw=1.7, label="Min-Var")
ax.plot(rv_vw.index, rv_vw.values, color=PAL["vw"], lw=1.7, ls="--", label="Val-Wgt")
ax.set_title("Rolling 12-month annualized volatility")
ax.set_ylabel("Vol")
ax.yaxis.set_major_formatter(PCT0)
ax.legend(loc="upper right")
year_axis(ax)

# ── Panel 3: drawdown ─────────────────────────────────────────────────────
ax = axes[1, 0]
def drawdown_series(rp):
    c = (1 + rp.dropna()).cumprod()
    return (c - c.cummax()) / c.cummax() * 100
dd_mv = drawdown_series(rp_mv); dd_vw = drawdown_series(rp_vw)
ax.fill_between(dd_vw.index, dd_vw.values, 0, alpha=0.18, color=PAL["vw"], label="Val-Wgt")
ax.fill_between(dd_mv.index, dd_mv.values, 0, alpha=0.45, color=PAL["mv"], label="Min-Var")
ax.plot(dd_vw.index, dd_vw.values, color=PAL["vw"], lw=0.8, ls="--")
ax.plot(dd_mv.index, dd_mv.values, color=PAL["mv"], lw=1.0)
# Annotate worst month for each
for s, c, lab in [(dd_mv, PAL["mv"], "MV"), (dd_vw, PAL["vw"], "VW")]:
    t_min = s.idxmin()
    ax.annotate(f"{lab}: {s.min():.1f}%",
                xy=(t_min, s.min()), xytext=(8, -8 if lab == "MV" else 8),
                textcoords="offset points", color=c, fontsize=8, fontweight="bold")
ax.set_title("Drawdown from running peak")
ax.set_ylabel("Drawdown")
ax.yaxis.set_major_formatter(PCT0)
ax.legend(loc="lower left")
year_axis(ax)

# ── Panel 4: universe size ────────────────────────────────────────────────
ax = axes[1, 1]
yrs = sorted(universe.keys())
sizes = [len(universe[y]) for y in yrs]
bars = ax.bar(yrs, sizes, color=PAL["mv"], alpha=0.78, edgecolor="white", width=0.7)
for x, n in zip(yrs, sizes):
    ax.text(x, n + 6, str(n), ha="center", fontsize=8, color=PAL["mv"])
ax.set_title("EUR investment-set size by year")
ax.set_ylabel("Eligible firms")
ax.set_xticks(yrs); ax.set_xticklabels(yrs, rotation=45, ha="right")
ax.grid(axis="y")
ax.set_axisbelow(True)
ax.spines["bottom"].set_visible(True)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part1_EUR_figures.{ext}")
plt.show()
print("   Saved: SAAM_Part1_EUR_figures.{pdf,png}")

### 14. Excel export — official template + extended results

Two output workbooks:

- `SAAM_Part1_EUR_template.xlsx` — fills the official project template (summary stats in B/C, monthly returns in F/G, embeds the cumulative-return chart at B9).
- `SAAM_Part1_EUR_results.xlsx` — extended dump: summary stats, monthly returns, year-by-year MV weights, top-10 holdings.

In [ ]:
print("\n[10] Exporting Excel files ...")

def template_stats(rp, rf_s):
    rp = rp.dropna()
    rf = rf_s.reindex(rp.index).ffill().fillna(0)
    T = len(rp)
    mu_arith = 12 * rp.mean()
    mu_geom  = (1 + rp).prod() ** (12 / T) - 1
    sig_ann  = rp.std() * np.sqrt(12)
    rf_ann   = 12 * rf.mean()
    SR = (mu_arith - rf_ann) / sig_ann
    return {"ann_avg_ret": mu_arith, "ann_vol": sig_ann, "ann_cum_ret": mu_geom,
            "sharpe": SR, "min_mo": rp.min(), "max_mo": rp.max()}

vw_stats = template_stats(rp_vw, rf_mon)
mv_stats = template_stats(rp_mv, rf_mon)

wb = load_workbook(TEMPLATE_PATH)
ws = wb["Sheet1"]
stat_keys = ["ann_avg_ret", "ann_vol", "ann_cum_ret", "sharpe", "min_mo", "max_mo"]
for i, key in enumerate(stat_keys):
    ws.cell(row=3 + i, column=2, value=round(vw_stats[key], 8))
    ws.cell(row=3 + i, column=3, value=round(mv_stats[key], 8))

# Embed cumulative-return chart in template
cum_plot_path = f"{OUT}SAAM_Part1_EUR_cumulative.png"
fig_cum, ax_cum = plt.subplots(figsize=(7, 4))
cm = (1 + rp_mv.dropna()).cumprod()
cv = (1 + rp_vw.dropna()).cumprod()
ax_cum.plot(cm.index, cm.values, color=PAL["mv"], lw=1.8, label=r"Min-Var $P_{oos}^{(mv)}$")
ax_cum.plot(cv.index, cv.values, color=PAL["vw"], lw=1.8, ls="--", label=r"Val-Wgt $P^{(vw)}$")
ax_cum.set_title("Cumulative return (base = 1, Jan 2014)")
ax_cum.set_ylabel("Wealth multiple")
ax_cum.legend()
year_axis(ax_cum)
fig_cum.tight_layout()
fig_cum.savefig(cum_plot_path)
plt.close(fig_cum)

img = XlImage(cum_plot_path); img.width = 500; img.height = 280
ws.add_image(img, "B9")

vw_by_ym = {(d.year, d.month): v for d, v in rp_vw.items()}
mv_by_ym = {(d.year, d.month): v for d, v in rp_mv.items()}
for row_idx in range(3, 3 + 144):
    date_cell = ws.cell(row=row_idx, column=5).value
    if date_cell is None:
        continue
    dt = pd.Timestamp(date_cell)
    ym = (dt.year, dt.month)
    vw_val = vw_by_ym.get(ym, np.nan); mv_val = mv_by_ym.get(ym, np.nan)
    ws.cell(row=row_idx, column=6, value=round(float(vw_val), 8) if not np.isnan(vw_val) else None)
    ws.cell(row=row_idx, column=7, value=round(float(mv_val), 8) if not np.isnan(mv_val) else None)

xlsx_out = f"{OUT}SAAM_Part1_EUR_template.xlsx"
wb.save(xlsx_out)
print(f"   Template saved: {xlsx_out}")

# Extended results workbook
xlsx_ext = f"{OUT}SAAM_Part1_EUR_results.xlsx"
with pd.ExcelWriter(xlsx_ext, engine="openpyxl") as writer:
    stats_df.to_excel(writer, sheet_name="Summary_Stats")
    ro = pd.DataFrame({"Min-Var": rp_mv, "Value-Weighted": rp_vw})
    ro.index = ro.index.strftime("%Y-%m")
    ro.to_excel(writer, sheet_name="Monthly_Returns")
    wd = pd.DataFrame(mv_w_dict).T.fillna(0)
    wd.index.name = "Year"
    wd.rename(columns=isin_name, inplace=True)
    wd.to_excel(writer, sheet_name="MV_Weights")
    rows = []
    for Y in sorted(mv_w_dict):
        for rk, (i, wt) in enumerate(mv_w_dict[Y].sort_values(ascending=False).head(10).items(), 1):
            cty = static.loc[static["ISIN"] == i, "Country"].values
            rows.append({"Year": Y, "Rank": rk, "ISIN": i,
                         "Name": isin_name.get(i, i),
                         "Country": cty[0] if len(cty) else "",
                         "Weight (%)": round(wt * 100, 3)})
    pd.DataFrame(rows).to_excel(writer, sheet_name="Top10_Holdings", index=False)
print(f"   Extended results saved: {xlsx_ext}")

---
## Part II — Carbon-Aware Portfolio Allocation

We now extend the Part I framework with explicit carbon constraints. Three carbon-aware portfolios:

| Portfolio | Objective | Constraint |
|---|---|---|
| `MV(0.5)` | $\min \alpha' \Sigma \alpha$ | $\mathrm{CF}(\alpha) \le 0.5 \cdot \mathrm{CF}(P^{(mv)})_Y$ |
| `VW(0.5)` | $\min (\alpha - w_{vw})' \Sigma (\alpha - w_{vw})$ | $\mathrm{CF}(\alpha) \le 0.5 \cdot \mathrm{CF}(P^{(vw)})_Y$ |
| `VW(NZ)` | $\min (\alpha - w_{vw})' \Sigma (\alpha - w_{vw})$ | $\mathrm{CF}(\alpha) \le (1-0.10)^{Y-2012} \cdot \mathrm{CF}(P^{(vw)})_{2013}$ |

All three are convex QPs solved via SLSQP with analytical Jacobians. Per-firm CF coefficient is $c_i = E_i / \mathrm{Cap}_i$. Per-year feasibility is checked ex ante via an LP (`min c'α s.t. e'α=1, 0 ≤ α ≤ 1`); if the project target falls below the LP minimum, we relax it to `(LP_min × 1.001)` and log it. **No relaxation was required in any of the three runs.**

### 15. Carbon data preparation and coverage diagnostics

- `co2_tot = co2_s1 + co2_s2` — total emissions (tCO₂)
- `rev_m = rev / 1000` — revenue converted from thousands → millions USD
- `CI_firm = co2_tot / rev_m` — per-firm carbon intensity (tCO₂ / M\$ rev)

Coverage: emissions are 100% by construction (filter 4 of universe). Revenue gaps mostly reflect holding companies that report emissions but not revenue; market-cap gaps similarly affect a few names. Both reduce the *coverage by weight*, but stay above 99% in every year.

In [ ]:
print("\n[11] Preparing carbon data ...")

co2_tot = co2_s1 + co2_s2
rev_m   = rev / 1000.0  # thousands → millions USD

with np.errstate(divide="ignore", invalid="ignore"):
    CI_firm = co2_tot / rev_m

diag = []
for Y in years_part2:
    isins = universe[Y]
    diag.append({
        "Y": Y, "Universe": len(isins),
        "Emissions": int(co2_tot.loc[isins, Y].notna().sum()) if Y in co2_tot.columns else 0,
        "Revenue":   int(rev_m.loc[isins, Y].notna().sum())   if Y in rev_m.columns else 0,
        "Cap_yr":    int(mv_y.loc[isins, Y].notna().sum())    if Y in mv_y.columns else 0,
    })
diag_df = pd.DataFrame(diag).set_index("Y")
display(diag_df)

miss_E = int((diag_df["Universe"] - diag_df["Emissions"]).sum())
assert miss_E == 0, f"Emissions missing in universe: {miss_E}"
print(f"✓ Emissions: 100% coverage in universe")
print(f"  Coverage gaps: Revenue missing {int((diag_df['Universe'] - diag_df['Revenue']).sum())} firm-yrs, "
      f"Cap missing {int((diag_df['Universe'] - diag_df['Cap_yr']).sum())}")

In [ ]:
def _firm_ci(isin, Y):
    """Per-firm CI (tCO2/M$ rev), NaN if revenue missing or zero."""
    if (isin in co2_tot.index and Y in co2_tot.columns
            and isin in rev_m.index and Y in rev_m.columns
            and pd.notna(rev_m.loc[isin, Y]) and rev_m.loc[isin, Y] > 0):
        return float(co2_tot.loc[isin, Y] / rev_m.loc[isin, Y])
    return np.nan

def most_avoided(w_port, w_bench, Y, n=10, title=""):
    """Firms most underweighted in w_port vs w_bench."""
    all_isins = w_bench.index[w_bench > 1e-6]
    rows = []
    for isin in all_isins:
        wb = float(w_bench.get(isin, 0.0))
        wp = float(w_port.get(isin, 0.0))
        cty = static.loc[static["ISIN"] == isin, "Country"].values
        rows.append({
            "Name":     isin_name.get(isin, isin),
            "Country":  cty[0] if len(cty) else "",
            "Bench (%)": round(wb * 100, 3),
            "Port (%)":  round(wp * 100, 3),
            "Active (pp)": round((wp - wb) * 100, 3),
            "CI (tCO2/M$rev)": (round(_firm_ci(isin, Y), 1)
                                if not np.isnan(_firm_ci(isin, Y)) else np.nan),
        })
    df = pd.DataFrame(rows).sort_values("Active (pp)").head(n).reset_index(drop=True)
    if title:
        print(f"\\n   MOST AVOIDED — {title}  (Y={Y}, top {n}):")
    return df

### 16. §3.1 — Baseline carbon profile (MV vs VW)

Two metrics, both standard PCAF/MSCI definitions:

- **WACI** $= \sum_i \alpha_i \cdot (E_i / \mathrm{Rev}_i)$ — weighted-average carbon intensity (tCO₂ / M\$ revenue), measures the *average emissions-per-dollar-of-revenue* of the portfolio's underlying businesses.
- **CF** $= \sum_i \alpha_i \cdot (E_i / \mathrm{Cap}_i)$ — carbon footprint with ownership attribution (tCO₂ / M\$ invested). For a value-weighted portfolio this collapses to $\sum E_i / \sum \mathrm{Cap}_i$ (verified below).

**What to expect from this baseline:**

> *From the findings:* Both portfolios show meaningful organic decarbonization. VW CF declines from 213.06 (2013) to 89.18 (2024) — **a 58% reduction with no policy intervention**, driven by efficiency gains at large fossil-fuel firms (Shell, TotalEnergies, ENI, Equinor) and utility transition (E.ON, Iberdrola, Enel). MV WACI is below VW WACI in 10 of 12 years — the natural low-vol tilt favors banks, insurance, healthcare and consumer staples, all low-emitter sectors.

In [ ]:
print("\n[12] Computing baseline portfolio carbon metrics ...")

def carbon_metrics(weights, Y):
    """WACI (tCO2/M$rev) and CF (tCO2/M$inv) for a portfolio at year-end Y."""
    isins = list(weights.index)
    w = weights.values.astype(float)
    E = co2_tot.loc[isins, Y].values.astype(float)
    R = rev_m.loc[isins, Y].values.astype(float)
    C = mv_y.loc[isins, Y].values.astype(float)

    with np.errstate(divide="ignore", invalid="ignore"):
        CI = np.where((R > 0) & np.isfinite(R), E / R, np.nan)
    valid_W = np.isfinite(CI)
    WACI  = float(np.nansum(w * np.where(valid_W, CI, 0.0)))
    cov_W = float(w[valid_W].sum())

    with np.errstate(divide="ignore", invalid="ignore"):
        ED = np.where((C > 0) & np.isfinite(C), E / C, np.nan)
    valid_F = np.isfinite(ED)
    CF    = float(np.nansum(w * np.where(valid_F, ED, 0.0)))
    cov_F = float(w[valid_F].sum())

    return {"WACI": WACI, "CF": CF, "cov_W": cov_W, "cov_F": cov_F}

mv_carbon = pd.DataFrame([{"Y": Y, **carbon_metrics(mv_w_dict[Y], Y)}
                          for Y in years_part2]).set_index("Y")
vw_carbon = pd.DataFrame([{"Y": Y, **carbon_metrics(vw_weights(Y), Y)}
                          for Y in years_part2]).set_index("Y")

print("\n   Min-Variance portfolio P^(mv)_oos:")
display(mv_carbon.round(2))
print("\n   Value-Weighted portfolio P^(vw):")
display(vw_carbon.round(2))

# Verification: VW CF (weighted form) = aggregate form (sum E / sum Cap) ----
print("\n[12b] Verification: VW CF weighted = aggregate form")
max_err = 0.0
for Y in years_part2:
    isins = universe[Y]
    cap_Y = mv_y.loc[isins, Y].fillna(0.0)
    E_Y   = co2_tot.loc[isins, Y].fillna(0.0)
    cf_agg = float(E_Y.sum() / cap_Y.sum()) if cap_Y.sum() > 0 else np.nan
    cf_w   = vw_carbon.loc[Y, "CF"]
    rel_err = abs(cf_agg - cf_w) / max(abs(cf_w), 1e-12)
    max_err = max(max_err, rel_err)
print(f"   max relative error across years: {max_err:.2e}  ✓")

mv_lower = (mv_carbon["WACI"] < vw_carbon["WACI"]).sum()
print(f"\n   MV WACI < VW WACI in {mv_lower} of {len(years_part2)} years "
      f"(low-vol tilts to low-carbon sectors)")

### 17. Top carbon contributors (concentration story)

This is the structural fact behind every "active position" you'll see in §3.2 and §3.3: VW WACI is **dominated by a handful of names** every year. Holcim, RWE, Heidelberg Materials, ENEL, Air Liquide, Maersk, Rio Tinto and a few others account for ~30–35% of WACI consistently.

**Concentration is rising over time** — top-10 share goes from 43.5% (2013) to 51.5% (2024). As the rest of the portfolio decarbonises, the few hard-to-abate names take up a larger share of what remains. This is precisely why the carbon constraint binds with a small number of named tilts in the passive portfolios.

In [ ]:
def top_n_by_CI(Y, n=10):
    isins = universe[Y]
    CI = (co2_tot.loc[isins, Y] / rev_m.loc[isins, Y]).dropna().sort_values(ascending=False)
    rows = []
    for rk, isin in enumerate(CI.head(n).index, 1):
        cty = static.loc[static['ISIN'] == isin, 'Country'].values
        rows.append({'Rank': rk, 'Firm': isin_name.get(isin, isin)[:40],
                     'Country': cty[0] if len(cty) else '',
                     'CI (tCO2/M$rev)': round(float(CI.loc[isin]), 1),
                     'VW weight (%)': round(float(vw_weights(Y).get(isin, 0)) * 100, 3)})
    return pd.DataFrame(rows).set_index('Rank')

def top_n_by_contrib(Y, weights, n=10, label='WACI'):
    isins = list(weights.index)
    CI = (co2_tot.loc[isins, Y] / rev_m.loc[isins, Y])
    contrib = (weights * CI).dropna().sort_values(ascending=False)
    rows = []
    for rk, isin in enumerate(contrib.head(n).index, 1):
        cty = static.loc[static['ISIN'] == isin, 'Country'].values
        rows.append({'Rank': rk, 'Firm': isin_name.get(isin, isin)[:40],
                     'Country': cty[0] if len(cty) else '',
                     'Weight (%)': round(float(weights.loc[isin]) * 100, 3),
                     'CI (tCO2/M$rev)': round(float(CI.loc[isin]), 1),
                     f'Contrib to {label}': round(float(contrib.loc[isin]), 2)})
    return pd.DataFrame(rows).set_index('Rank')

snapshot_years = [2013, 2018, 2024]

for Y in snapshot_years:
    print(f"\n— Top-10 carbon-intensive firms in EUR universe, Y={Y} —")
    display(top_n_by_CI(Y, 10))

for Y in snapshot_years:
    print(f"\n— Top-10 contributors to VW WACI, Y={Y} —")
    display(top_n_by_contrib(Y, vw_weights(Y), 10, label='VW WACI'))

### 18. §3.1 figure — WACI and CF over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

for ax, col, ylab, title in [
    (axes[0], "WACI", r"tCO$_2$ / M\$ revenue", "WACI — weighted-average carbon intensity"),
    (axes[1], "CF",   r"tCO$_2$ / M\$ invested", "CF — carbon footprint (ownership-attributed)"),
]:
    ax.plot(mv_carbon.index, mv_carbon[col], "o-", color=PAL["mv"], lw=1.9,
            label=r"Min-Var $P^{(mv)}_{oos}$", markersize=5)
    ax.plot(vw_carbon.index, vw_carbon[col], "s--", color=PAL["vw"], lw=1.9,
            label=r"Val-Wgt $P^{(vw)}$", markersize=5)
    ax.set_title(title)
    ax.set_xlabel("Year"); ax.set_ylabel(ylab)
    ax.legend()

# Annotation: 58% organic VW decarbonization (Findings §3)
ax = axes[1]
y0, y1 = vw_carbon["CF"].iloc[0], vw_carbon["CF"].iloc[-1]
pct = (y1 / y0 - 1) * 100
ax.annotate(f"VW CF: {y0:.0f} → {y1:.0f}\n({pct:+.0f}% organic)",
            xy=(vw_carbon.index[-1], y1), xytext=(-90, 50),
            textcoords="offset points",
            fontsize=9, color=PAL["vw"], fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=PAL["vw"], lw=0.8))

fig.suptitle("§3.1 — Baseline carbon profile (MV vs VW)", fontsize=13, fontweight="bold")
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_carbon_baseline.{ext}")
plt.show()
print("   Saved: SAAM_Part2_carbon_baseline.{pdf,png}")

---
### 19. §3.2 — Active investor: $P^{(mv)}_{oos}(0.5)$

**Construction.** Same long-only GMV problem as §3.1, with one extra inequality constraint:
$$
\sum_i \alpha_i \cdot \tfrac{E_i}{\mathrm{Cap}_i} \,\le\, 0.5 \cdot \mathrm{CF}\!\left(P^{(mv)}_{oos}\right)_Y.
$$

The target is **path-dependent**: it's half of the *unconstrained* MV portfolio's CF for the same year. By construction the unconstrained MV solution has $\mathrm{CF} = 2 \times \mathrm{target}$, so the constraint binds in every year (slack = 0). Per-firm CF coefficient $c_i = E_i / \mathrm{Cap}_i$; firms with missing market cap are forced to $\alpha_i = 0$ via $c_i = +\infty$.

**Two caveats worth flagging up front (from the findings doc):**

> 1. **In 2014 and 2016 the relative target was *looser* than the VW benchmark.** MV CF was an outlier in those years (concentrated in a handful of high-CF mid-caps), so 0.5×MV happens to exceed VW CF. $P^{(mv)}(0.5)$ is therefore *more* carbon-intensive than VW in those two years. This is a property of relative targeting, not an error.
> 2. **Counter-intuitive removals.** In 2018, $P^{(mv)}(0.5)$ drops Coloplast (CI = 21) and Chubb (CI = 2) — both low-emitters. The constrained QP just has a different optimum on a different surface; it re-weights for variance reasons, not as a "delete the emitters" filter.

In [ ]:
print("\n[13] §3.2 — MV(0.5) optimisation ...")

cf_target_mv = (mv_carbon["CF"] * 0.5).rename("target_CF").copy()

def cf_vector(isins, Y):
    """c_i = E_i,Y / Cap_i,Y; +inf where cap missing — forces alpha_i = 0."""
    E = co2_tot.loc[isins, Y].values.astype(float)
    C = mv_y.loc[isins, Y].values.astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        c = np.where((C > 0) & np.isfinite(C) & np.isfinite(E), E / C, np.inf)
    return c, np.isfinite(c)

# Cached per-year LP minimum CF (for feasibility checks across §3.2, §3.3, §4.1)
_mfcf_cache = {}
def get_mfcf(Y):
    if Y not in _mfcf_cache:
        isins = universe[Y]
        c, valid = cf_vector(isins, Y)
        N = len(isins)
        if not valid.any():
            _mfcf_cache[Y] = np.nan
        else:
            c_lp = np.where(valid, c, 1e20)
            res = linprog(c=c_lp, A_eq=np.ones((1, N)), b_eq=[1.0],
                          bounds=[(0.0, 1.0)] * N, method="highs")
            _mfcf_cache[Y] = float(c_lp @ res.x) if res.success else np.nan
    return _mfcf_cache[Y]

def feasibility_check(cf_targets, label):
    adj, infeas = {}, []
    for Y in years_part2:
        cf_min = get_mfcf(Y)
        target_raw = float(cf_targets.loc[Y])
        if cf_min > target_raw:
            target_eff = cf_min * 1.001
            infeas.append({"Y": Y, "target_raw": target_raw,
                           "cf_min": cf_min, "target_eff": target_eff})
            adj[Y] = target_eff
        else:
            adj[Y] = target_raw
    if infeas:
        print(f"   ⚠ [{label}] Infeasibility — targets relaxed:")
        display(pd.DataFrame(infeas).round(3).set_index("Y"))
    else:
        print(f"   ✓ [{label}] All targets feasible")
    return adj

adj_targets = feasibility_check(cf_target_mv, "MV05")

In [ ]:
def min_var_cf_constrained(Sigma, c, target, isins, Y, prev_drift=None):
    """min α'Σα  s.t. α'e=1, c'α≤target, α≥0, α_i=0 if cap missing."""
    N = Sigma.shape[0]
    valid = np.isfinite(c)
    invalid = ~valid

    if prev_drift is not None:
        w0 = np.array([prev_drift.get(i, 0.0) for i in isins])
        w0[invalid] = 0.0
        s = w0.sum()
        w0 = w0 / s if s > 0 else np.where(valid, 1.0, 0.0) / max(valid.sum(), 1)
    else:
        w0 = np.where(valid, 1.0, 0.0) / max(valid.sum(), 1)

    bounds = [(0.0, 0.0) if invalid[i] else (0.0, 1.0) for i in range(N)]
    c_safe = np.where(valid, c, 0.0)
    constraints = [
        {"type": "eq",   "fun": lambda w: w.sum() - 1.0,        "jac": lambda w: np.ones(N)},
        {"type": "ineq", "fun": lambda w: target - c_safe @ w,  "jac": lambda w: -c_safe},
    ]
    return minimize(
        fun=lambda w: float(w @ Sigma @ w),
        x0=w0, jac=lambda w: 2.0 * (Sigma @ w),
        method="SLSQP", bounds=bounds, constraints=constraints,
        options={"ftol": 1e-10, "maxiter": 2000},
    )

mv05_w_dict, mv05_ret, _mv05_drift, solver_log = {}, {}, {}, []

for Y in years_part2:
    eligible = universe[Y]
    target = adj_targets[Y]
    _, Sig = get_cov(Y)
    c, valid = cf_vector(eligible, Y)

    res = min_var_cf_constrained(Sig, c, target, eligible, Y,
                                 prev_drift=_mv05_drift.get(Y))
    w = np.clip(res.x, 0.0, 1.0)
    if w.sum() > 0:
        w /= w.sum()

    cf_real = float(np.where(valid, c, 0.0) @ w)
    violates = cf_real > target * (1 + 1e-6)
    if violates:
        print(f"   ⚠ Y={Y}: CF constraint violated ({cf_real:.3f} > {target:.3f})")

    solver_log.append({"Y": Y, "ok": bool(res.success), "iter": int(res.nit),
                       "ann_var": float(w @ Sig @ w), "target": target,
                       "cf_real": cf_real,
                       "binds": abs(cf_real - target) / max(abs(target), 1e-12) < 1e-3,
                       "violates": bool(violates),
                       "n_active": int((w > 1e-6).sum())})

    mv05_w_dict[Y] = pd.Series(w, index=eligible)

    R_next = get_oos_returns(Y)
    next_months = _months_of[Y + 1]
    ww = w.copy()
    port_ret = []
    for t in next_months:
        r_t = R_next[t].values
        rp_t = float(ww @ r_t)
        port_ret.append(rp_t)
        ww = ww * (1.0 + r_t) / max(1.0 + rp_t, 1e-12)
    mv05_ret[Y + 1] = pd.Series(port_ret, index=next_months)
    _mv05_drift[Y + 1] = dict(zip(eligible, ww))

rp_mv05 = pd.concat(mv05_ret).droplevel(0).sort_index()
rp_mv05.index = pd.DatetimeIndex(rp_mv05.index)

solver_df = pd.DataFrame(solver_log).set_index("Y")
print("\n   Solver / constraint diagnostics:")
display(solver_df.round(3))

In [ ]:
# Verification — MV05
for Y, w in mv05_w_dict.items():
    assert abs(w.sum() - 1.0) < 1e-6 and (w >= -1e-8).all()

mv05_carbon = pd.DataFrame(
    [{"Y": Y, **carbon_metrics(mv05_w_dict[Y], Y)} for Y in years_part2]
).set_index("Y")

cf_check = pd.DataFrame({
    "target_raw":   cf_target_mv,
    "target_eff":   pd.Series(adj_targets),
    "CF_realized":  mv05_carbon["CF"],
    "WACI_realized": mv05_carbon["WACI"],
})
cf_check["slack_vs_eff"] = cf_check["target_eff"] - cf_check["CF_realized"]
print("\n   CF realized vs target — every year, constraint binds (slack ≈ 0):")
display(cf_check.round(3))

# Ex-ante variance cost of constraint
var_rows = []
for Y in years_part2:
    _, Sig = get_cov(Y)
    var_rows.append({"Y": Y,
                     "var_MV":   float(mv_w_dict[Y].values   @ Sig @ mv_w_dict[Y].values),
                     "var_MV05": float(mv05_w_dict[Y].values @ Sig @ mv05_w_dict[Y].values)})
var_df = pd.DataFrame(var_rows).set_index("Y")
var_df["sigma_MV"]   = np.sqrt(var_df["var_MV"]) * np.sqrt(12) * 100
var_df["sigma_MV05"] = np.sqrt(var_df["var_MV05"]) * np.sqrt(12) * 100
var_df["delta_pp"]   = var_df["sigma_MV05"] - var_df["sigma_MV"]
print("\n   Ex-ante annualized vol cost of carbon constraint (pp):")
display(var_df[["sigma_MV", "sigma_MV05", "delta_pp"]].round(3))

stats_32 = pd.DataFrame([
    compute_perf(rp_mv,    rf_mon, "P^(mv)_oos"),
    compute_perf(rp_mv05,  rf_mon, "P^(mv)_oos(0.5)"),
    compute_perf(rp_vw,    rf_mon, "P^(vw) (benchmark)"),
]).set_index("Portfolio")
print("\n   Performance — MV vs MV(0.5):")
display(stats_32)

**Composition shifts** (snapshot years 2013, 2018, 2024). The "Removed" / "Added" lists in some years contain firms whose CI is *low* — this is the variance-driven re-weighting on a different feasible surface, not a "delete the emitters" filter (see caveat above).

In [ ]:
def composition_diff(Y, top_n=5):
    w_mv  = mv_w_dict[Y].reindex(universe[Y]).fillna(0.0)
    w_mv5 = mv05_w_dict[Y].reindex(universe[Y]).fillna(0.0)
    diff = (w_mv5 - w_mv).sort_values()
    drops = diff.head(top_n).iloc[::-1]
    adds  = diff.tail(top_n).iloc[::-1]

    rows = []
    for kind, src in [('Removed', diff.head(top_n)), ('Added', adds)]:
        for isin, dlt in src.items():
            ci = _firm_ci(isin, Y)
            rows.append({'Y': Y, 'Direction': kind,
                         'Firm': isin_name.get(isin, isin)[:40],
                         'Δ weight (pp)': round(dlt * 100, 2),
                         'CI (tCO2/M$rev)': round(ci, 1) if not np.isnan(ci) else np.nan})
    return pd.DataFrame(rows)

for Y in snapshot_years:
    print(f"\n— §3.2 composition shifts MV → MV(0.5), Y={Y} —")
    display(composition_diff(Y, top_n=5))

# Most-avoided position table for snapshot year(s)
print("\n— Most-avoided positions (MV(0.5) vs MV) —")
for Y in snapshot_years:
    df = most_avoided(mv05_w_dict[Y].reindex(universe[Y]).fillna(0.0),
                      mv_w_dict[Y].reindex(universe[Y]).fillna(0.0),
                      Y, n=10, title="firms cut most by carbon constraint")
    print(f"\n   Y={Y}")
    display(df)

### 20. §3.2 figure — MV vs MV(0.5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("§3.2 — MV vs MV(0.5)  (active investor, 50% CF cap on its own baseline)",
             fontsize=13, fontweight="bold", y=1.00)

# ── Panel 1: cumulative return ────────────────────────────────────────────
ax = axes[0, 0]
for rp, color, ls, lab in [(rp_mv,   PAL["mv"],   "-",  r"MV $P^{(mv)}_{oos}$"),
                           (rp_mv05, PAL["mv05"], "-",  r"MV(0.5)"),
                           (rp_vw,   PAL["vw"],   "--", r"VW $P^{(vw)}$")]:
    cum = (1 + rp.dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=color, ls=ls, lw=1.8, label=lab)
    label_endpoint(ax, cum.index[-1], cum.iloc[-1], f" {cum.iloc[-1]:.2f}×", color)
ax.set_title("Cumulative return (base = 1, Jan 2014)")
ax.set_ylabel("Wealth multiple")
ax.axhline(1, color="#BBBBBB", lw=0.6, zorder=0)
shade_events(ax, label_first=True)
ax.legend(loc="upper left")
year_axis(ax)

# ── Panel 2: drawdown ─────────────────────────────────────────────────────
ax = axes[0, 1]
def dd(rp):
    c = (1 + rp.dropna()).cumprod()
    return (c - c.cummax()) / c.cummax() * 100
ax.fill_between(dd(rp_mv).index,   dd(rp_mv).values,   0, alpha=0.30, color=PAL["mv"])
ax.fill_between(dd(rp_mv05).index, dd(rp_mv05).values, 0, alpha=0.30, color=PAL["mv05"])
ax.plot(dd(rp_mv).index,   dd(rp_mv).values,   color=PAL["mv"],   lw=1.0, label="MV")
ax.plot(dd(rp_mv05).index, dd(rp_mv05).values, color=PAL["mv05"], lw=1.0, label="MV(0.5)")
ax.set_title("Drawdown from running peak")
ax.set_ylabel("Drawdown")
ax.yaxis.set_major_formatter(PCT0)
ax.legend(loc="lower left")
year_axis(ax)

# ── Panel 3: WACI ─────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(mv_carbon.index,   mv_carbon["WACI"],   "o-", color=PAL["mv"],   label="MV", lw=1.7)
ax.plot(mv05_carbon.index, mv05_carbon["WACI"], "s-", color=PAL["mv05"], label="MV(0.5)", lw=1.7)
ax.plot(vw_carbon.index,   vw_carbon["WACI"],   "x--", color=PAL["vw"],  label="VW", lw=1.4)
ax.set_title("WACI evolution")
ax.set_ylabel(r"tCO$_2$ / M\$ rev")
ax.legend()

# ── Panel 4: CF + 50% target ──────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(mv_carbon.index,   mv_carbon["CF"],   "o-",  color=PAL["mv"],   label="MV", lw=1.7)
ax.plot(mv05_carbon.index, mv05_carbon["CF"], "s-",  color=PAL["mv05"], label="MV(0.5)", lw=1.7)
ax.plot(cf_target_mv.index, cf_target_mv.values, ":", color=PAL["target"], lw=1.3,
        label="0.5×CF(MV) target")
ax.plot(vw_carbon.index,   vw_carbon["CF"],   "x--", color=PAL["vw"],  label="VW", lw=1.4)
# Highlight 2014 and 2016 — relative target looser than VW (caveat)
for yr in [2014, 2016]:
    if yr in mv_carbon.index:
        ax.scatter([yr], [mv05_carbon.loc[yr, "CF"]], s=140, facecolors="none",
                   edgecolors=PAL["high_e"], lw=1.4, zorder=5)
ax.text(0.02, 0.04,
        "Circled: years where 0.5×MV target > VW CF\n(relative target is path-dependent)",
        transform=ax.transAxes, fontsize=8, color=PAL["high_e"], style="italic",
        verticalalignment="bottom")
ax.set_title("CF evolution + 50%-of-MV target")
ax.set_ylabel(r"tCO$_2$ / M\$ inv")
ax.legend()

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_section32.{ext}")
plt.show()
print("   Saved: SAAM_Part2_section32.{pdf,png}")

---
### 21. §3.3 — Passive investor: $P^{(vw)}_{oos}(0.5)$

**Construction.** Tracking-error minimisation against the value-weighted benchmark, with the same 50% CF cap on the *benchmark*'s CF:
$$
\min_\alpha (\alpha - w_{vw})' \Sigma (\alpha - w_{vw}) \quad
\text{s.t.} \quad \alpha' \mathbf{e} = 1, \quad
\sum_i \alpha_i \cdot \tfrac{E_i}{\mathrm{Cap}_i} \le 0.5 \cdot \mathrm{CF}(P^{(vw)})_Y,\quad
\alpha \ge 0.
$$

The optimisation is warm-started at $w_{vw}$ — infeasible at the start (its CF is above target), feasible at the optimum. The constraint binds in every year.

**The same names get hit every year.** Holcim is the largest cut in all three snapshot years studied; Heidelberg Materials, RWE, ENEL, A.P. Møller-Mærsk show up consistently. Tilts are individually small (largest single position change is −0.34 pp on Holcim 2018) because the TE objective spreads the constraint over hundreds of names. **The aggregate active position is a clear sector tilt — long staples / healthcare / financials, short utilities / energy / materials** (visualised in §22 below).

In [ ]:
print("\n[14] §3.3 — VW(0.5) optimisation ...")

cf_target_vw = (vw_carbon["CF"] * 0.5).rename("target_CF").copy()
adj_targets_vw = feasibility_check(cf_target_vw, "VW05")

def min_te_cf_constrained(Sigma, w_bench, c, target, isins, warm_start=None):
    """min (α-w_b)'Σ(α-w_b)  s.t. e'α=1, c'α≤target, α≥0."""
    N = Sigma.shape[0]
    valid = np.isfinite(c)
    invalid = ~valid
    w0 = (warm_start if warm_start is not None else w_bench.copy())
    w0[invalid] = 0.0
    s = w0.sum()
    w0 = w0 / s if s > 0 else np.where(valid, 1.0, 0.0) / max(valid.sum(), 1)

    bounds = [(0.0, 0.0) if invalid[i] else (0.0, 1.0) for i in range(N)]
    c_safe = np.where(valid, c, 0.0)
    constraints = [
        {"type": "eq",   "fun": lambda w: w.sum() - 1.0,         "jac": lambda w: np.ones(N)},
        {"type": "ineq", "fun": lambda w: target - c_safe @ w,   "jac": lambda w: -c_safe},
    ]
    return minimize(
        fun=lambda w: float((w - w_bench) @ Sigma @ (w - w_bench)),
        x0=w0, jac=lambda w: 2.0 * (Sigma @ (w - w_bench)),
        method="SLSQP", bounds=bounds, constraints=constraints,
        options={"ftol": 1e-10, "maxiter": 3000},
    )

vw05_w_dict, vw05_ret, solver_log_vw = {}, {}, []

for Y in years_part2:
    eligible = universe[Y]
    target = adj_targets_vw[Y]
    _, Sig = get_cov(Y)
    c, valid = cf_vector(eligible, Y)
    w_bench = vw_weights(Y).reindex(eligible).fillna(0.0).values

    res = min_te_cf_constrained(Sig, w_bench, c, target, eligible)
    w = np.clip(res.x, 0.0, 1.0)
    if w.sum() > 0:
        w /= w.sum()

    cf_real = float(np.where(valid, c, 0.0) @ w)
    te2 = float((w - w_bench) @ Sig @ (w - w_bench))
    violates = cf_real > target * (1 + 1e-6)
    if violates:
        print(f"   ⚠ Y={Y}: CF violated ({cf_real:.3f} > {target:.3f})")
    if not res.success:
        print(f"   ⚠ Y={Y}: SLSQP did not converge — {res.message}")

    solver_log_vw.append({"Y": Y, "ok": bool(res.success), "iter": int(res.nit),
                          "ann_TE_pct": np.sqrt(max(te2, 0.0)) * np.sqrt(12) * 100,
                          "target": target, "cf_real": cf_real,
                          "binds": abs(cf_real - target) / max(abs(target), 1e-12) < 1e-3,
                          "violates": bool(violates),
                          "n_active": int((w > 1e-6).sum())})

    vw05_w_dict[Y] = pd.Series(w, index=eligible)

    R_next = get_oos_returns(Y)
    next_months = _months_of[Y + 1]
    ww = w.copy()
    port_ret = []
    for t in next_months:
        r_t = R_next[t].values
        rp_t = float(ww @ r_t)
        port_ret.append(rp_t)
        ww = ww * (1.0 + r_t) / max(1.0 + rp_t, 1e-12)
    vw05_ret[Y + 1] = pd.Series(port_ret, index=next_months)

rp_vw05 = pd.concat(vw05_ret).droplevel(0).sort_index()
rp_vw05.index = pd.DatetimeIndex(rp_vw05.index)

solver_df_vw = pd.DataFrame(solver_log_vw).set_index("Y")
print("\n   Solver / TE diagnostics — note the tiny ex-ante TE ('ann_TE_pct'); "
      "we will diagnose the 16× ex-ante↔realized gap in §24 below.")
display(solver_df_vw.round(3))

In [ ]:
# Verification — VW05
for Y, w in vw05_w_dict.items():
    assert abs(w.sum() - 1.0) < 1e-6 and (w >= -1e-8).all()

vw05_carbon = pd.DataFrame(
    [{"Y": Y, **carbon_metrics(vw05_w_dict[Y], Y)} for Y in years_part2]
).set_index("Y")

cf_check_vw = pd.DataFrame({
    "target_eff":   pd.Series(adj_targets_vw),
    "CF_realized":  vw05_carbon["CF"],
    "WACI_realized": vw05_carbon["WACI"],
})
cf_check_vw["slack"] = cf_check_vw["target_eff"] - cf_check_vw["CF_realized"]
print("   CF realized vs target:")
display(cf_check_vw.round(3))

te_ep = (rp_vw05.dropna() - rp_vw.dropna()).std() * np.sqrt(12) * 100
print(f"\n   Realized annualized TE of VW(0.5) vs VW = {te_ep:.2f}%")

stats_33 = pd.DataFrame([
    compute_perf(rp_vw,    rf_mon, "P^(vw) (benchmark)"),
    compute_perf(rp_vw05,  rf_mon, "P^(vw)_oos(0.5)"),
    compute_perf(rp_mv05,  rf_mon, "P^(mv)_oos(0.5) [reference]"),
]).set_index("Portfolio")
display(stats_33)

ar = (rp_vw05 - rp_vw).dropna()
ir = ar.mean() * 12 / (ar.std() * np.sqrt(12)) if ar.std() > 0 else np.nan
print(f"\n   IR (VW(0.5) vs VW) = {ir:.3f}  |  AR = {ar.mean()*12*100:.2f}%/yr  |  TE = {te_ep:.2f}%")

# Standard error on annualized active return: TE/sqrt(years)
se_ar = te_ep / np.sqrt(len(ar)/12)
t_stat = ar.mean() * 12 * 100 / se_ar if se_ar > 0 else np.nan
print(f"   Standard error on AR ≈ {se_ar:.2f}%/yr  →  t = {t_stat:.2f}  "
      f"({'significant' if abs(t_stat) > 1.96 else 'NOT significant at 5%'})")

### 22. §3.3 figure — VW vs VW(0.5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("§3.3 — VW vs VW(0.5)  (passive investor, 50% CF cap on benchmark)",
             fontsize=13, fontweight="bold", y=1.00)

# ── Panel 1: cumulative return ────────────────────────────────────────────
ax = axes[0, 0]
for rp, color, ls, lab in [
    (rp_vw,   PAL["vw"],   "--", r"VW $P^{(vw)}$"),
    (rp_vw05, PAL["vw05"], "-",  r"VW(0.5)"),
    (rp_mv05, PAL["mv05"], ":",  r"MV(0.5) [reference]"),
]:
    cum = (1 + rp.dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=color, ls=ls, lw=1.8, label=lab)
    label_endpoint(ax, cum.index[-1], cum.iloc[-1], f" {cum.iloc[-1]:.2f}×", color)
ax.set_title("Cumulative return (base = 1, Jan 2014)")
ax.set_ylabel("Wealth multiple")
ax.axhline(1, color="#BBBBBB", lw=0.6, zorder=0)
shade_events(ax, label_first=True)
ax.legend(loc="upper left")
year_axis(ax)

# ── Panel 2: cumulative active return ──────────────────────────────────────
ax = axes[0, 1]
ar_cum = (1 + ar).cumprod()
ax.plot(ar_cum.index, (ar_cum - 1) * 100, color=PAL["vw05"], lw=1.6)
ax.fill_between(ar_cum.index, (ar_cum - 1) * 100, 0,
                color=PAL["vw05"], alpha=0.18)
ax.axhline(0, color="k", lw=0.6)
shade_events(ax, label_first=False)
ax.set_title(f"Cumulative active return: VW(0.5) − VW   (IR = {ir:+.2f})")
ax.set_ylabel("Cumulative active return")
ax.yaxis.set_major_formatter(PCT0)
year_axis(ax)
# Annotation: drag concentrated in 2021-22 (per findings)
ax.annotate("Drag concentrated\nin 2021–22\nenergy bull", xy=(pd.Timestamp("2022-06-01"), -3),
            xytext=(20, -30), textcoords="offset points",
            fontsize=8.5, color="#7A4017", style="italic",
            arrowprops=dict(arrowstyle="->", color="#7A4017", lw=0.8))

# ── Panel 3: WACI ─────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(vw_carbon.index,   vw_carbon["WACI"],   "x--", color=PAL["vw"],   label="VW", lw=1.5)
ax.plot(vw05_carbon.index, vw05_carbon["WACI"], "s-",  color=PAL["vw05"], label="VW(0.5)", lw=1.7)
ax.set_title("WACI evolution")
ax.set_ylabel(r"tCO$_2$ / M\$ rev")
ax.legend()

# ── Panel 4: CF + target ──────────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(vw_carbon.index,   vw_carbon["CF"],   "x--", color=PAL["vw"],   label="VW", lw=1.5)
ax.plot(vw05_carbon.index, vw05_carbon["CF"], "s-",  color=PAL["vw05"], label="VW(0.5)", lw=1.7)
ax.plot(cf_target_vw.index, cf_target_vw.values, ":", color=PAL["target"], lw=1.3,
        label="0.5×CF(VW) target")
ax.set_title("CF evolution + 50% target")
ax.set_ylabel(r"tCO$_2$ / M\$ inv")
ax.legend()

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_section33.{ext}")
plt.show()
print("   Saved: SAAM_Part2_section33.{pdf,png}")

---
### 23. §4.1 — Net-zero glide path: $P^{(vw)}_{oos}(NZ)$

**Construction.** Same TE-min objective as §3.3, but the carbon target now follows a *fixed* glide path anchored at the 2013 VW baseline:
$$
\mathrm{CF}(\alpha) \,\le\, (1 - \theta)^{Y - 2012} \cdot \mathrm{CF}(P^{(vw)})_{2013}, \quad \theta = 0.10.
$$

The exponent $Y - 2012$ means 2013 already imposes a 10% cut. By 2024 (12 years in), CF must be at most 28.2% of the 2013 baseline.

> **The surprise.** Over our 2014–2025 window, the NZ target was *looser* than 0.5×VW in **every single year**. The benchmark itself decarbonized organically by 58%, so a relative target inherits the structural transition while an absolute glide path stays fixed. *Net-zero pledges anchored to absolute historical baselines silently weaken when their benchmark decarbonizes faster than the pledged rate* — see §28 for the visual.

In [ ]:
print("\n[15] §4.1 — Net-zero glide path optimisation ...")

THETA = 0.10
CF_VW_2013 = float(vw_carbon.loc[2013, "CF"])

def nz_target(Y):
    return (1.0 - THETA) ** (Y - 2013 + 1) * CF_VW_2013

cf_target_nz = pd.Series({Y: nz_target(Y) for Y in years_part2}, name="target_CF")

target_compare = pd.DataFrame({
    "VW_realized":   vw_carbon["CF"],
    "0.5xVW_target": cf_target_vw,
    "NZ_target":     cf_target_nz,
})
target_compare["NZ_pct_of_2013"] = target_compare["NZ_target"] / CF_VW_2013 * 100
target_compare["NZ_tighter_than_0.5xVW"] = target_compare["NZ_target"] < target_compare["0.5xVW_target"]

print("\n   Glide path comparison:")
display(target_compare.round(2))
print(f"\n   Years where NZ is tighter than 0.5×VW: "
      f"{int(target_compare['NZ_tighter_than_0.5xVW'].sum())} / {len(years_part2)}")

adj_targets_nz = feasibility_check(cf_target_nz, "VWNZ")

# Optimization
vwnz_w_dict, vwnz_ret, solver_log_nz = {}, {}, []
for Y in years_part2:
    eligible = universe[Y]
    target = adj_targets_nz[Y]
    _, Sig = get_cov(Y)
    c, valid = cf_vector(eligible, Y)
    w_bench = vw_weights(Y).reindex(eligible).fillna(0.0).values

    res = min_te_cf_constrained(Sig, w_bench, c, target, eligible)
    w = np.clip(res.x, 0.0, 1.0)
    if w.sum() > 0:
        w /= w.sum()

    cf_real = float(np.where(valid, c, 0.0) @ w)
    te2 = float((w - w_bench) @ Sig @ (w - w_bench))
    violates = cf_real > target * (1 + 1e-6)

    solver_log_nz.append({"Y": Y, "ok": bool(res.success), "iter": int(res.nit),
                          "ann_TE_pct": np.sqrt(max(te2, 0.0)) * np.sqrt(12) * 100,
                          "target": target, "cf_real": cf_real,
                          "binds": abs(cf_real - target) / max(abs(target), 1e-12) < 1e-3,
                          "violates": bool(violates)})
    vwnz_w_dict[Y] = pd.Series(w, index=eligible)

    R_next = get_oos_returns(Y)
    next_months = _months_of[Y + 1]
    ww = w.copy()
    port_ret = []
    for t in next_months:
        r_t = R_next[t].values
        rp_t = float(ww @ r_t)
        port_ret.append(rp_t)
        ww = ww * (1.0 + r_t) / max(1.0 + rp_t, 1e-12)
    vwnz_ret[Y + 1] = pd.Series(port_ret, index=next_months)

rp_vwnz = pd.concat(vwnz_ret).droplevel(0).sort_index()
rp_vwnz.index = pd.DatetimeIndex(rp_vwnz.index)

solver_df_nz = pd.DataFrame(solver_log_nz).set_index("Y")
display(solver_df_nz.round(3))

In [ ]:
# Verification — VWNZ
for Y, w in vwnz_w_dict.items():
    assert abs(w.sum() - 1.0) < 1e-6 and (w >= -1e-8).all()

vwnz_carbon = pd.DataFrame(
    [{"Y": Y, **carbon_metrics(vwnz_w_dict[Y], Y)} for Y in years_part2]
).set_index("Y")
cf_check_nz = pd.DataFrame({
    "target_eff":   pd.Series(adj_targets_nz),
    "CF_realized":  vwnz_carbon["CF"],
    "WACI_realized": vwnz_carbon["WACI"],
    "pct_of_2013_CF": vwnz_carbon["CF"] / CF_VW_2013 * 100,
})
cf_check_nz["slack"] = cf_check_nz["target_eff"] - cf_check_nz["CF_realized"]
display(cf_check_nz.round(3))

final_pct = cf_check_nz.loc[2024, "pct_of_2013_CF"]
expected_pct = (1 - THETA) ** (2024 - 2013 + 1) * 100
print(f"\n   2024 CF as % of 2013 baseline = {final_pct:.1f}%  (target {expected_pct:.1f}%)")
print(f"   {'✓' if final_pct <= expected_pct + 0.5 else '⚠'} on/under glide path")

te_ep_nz = (rp_vwnz.dropna() - rp_vw.dropna()).std() * np.sqrt(12) * 100
print(f"\n   Realized annualized TE of VW(NZ) vs VW = {te_ep_nz:.2f}%")

# Performance + IR
stats_41 = pd.DataFrame([
    compute_perf(rp_vw,    rf_mon, "P^(vw)"),
    compute_perf(rp_vw05,  rf_mon, "P^(vw)_oos(0.5)"),
    compute_perf(rp_vwnz,  rf_mon, "P^(vw)_oos(NZ)"),
]).set_index("Portfolio")
display(stats_41)

ar_nz = (rp_vwnz - rp_vw).dropna()
te_nz_realized = ar_nz.std() * np.sqrt(12)
ir_nz = (ar_nz.mean() * 12) / te_nz_realized if te_nz_realized > 0 else np.nan
print(f"\n   IR (VW(NZ) vs VW)  = {ir_nz:+.3f}  AR={ar_nz.mean()*12*100:+.2f}%  TE={te_nz_realized*100:.2f}%")
print(f"   IR (VW(0.5) vs VW) = {ir:+.3f}  AR={ar.mean()*12*100:+.2f}%  TE={te_ep:.2f}%")

all_carbon = pd.DataFrame({
    "VW_CF":   vw_carbon["CF"],   "VW05_CF": vw05_carbon["CF"], "VWNZ_CF": vwnz_carbon["CF"],
    "VW_WACI": vw_carbon["WACI"], "VW05_WACI": vw05_carbon["WACI"], "VWNZ_WACI": vwnz_carbon["WACI"],
})
print("\n   Cumulative carbon footprint (2013–2024):")
print(f"     VW:     {vw_carbon['CF'].sum():7.1f}")
print(f"     VW(0.5):{vw05_carbon['CF'].sum():7.1f}  ({vw05_carbon['CF'].sum()/vw_carbon['CF'].sum()*100:.1f}% of VW)")
print(f"     VW(NZ): {vwnz_carbon['CF'].sum():7.1f}  ({vwnz_carbon['CF'].sum()/vw_carbon['CF'].sum()*100:.1f}% of VW)")

### 24. §4.1 figure — VW vs VW(0.5) vs VW(NZ)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("§4.1 — VW vs VW(0.5) vs VW(NZ)  (10%/yr glide path from 2013 baseline)",
             fontsize=13, fontweight="bold", y=1.00)

# ── Panel 1: cumulative return ────────────────────────────────────────────
ax = axes[0, 0]
for rp, color, ls, lab in [
    (rp_vw,   PAL["vw"],   "--", r"VW $P^{(vw)}$"),
    (rp_vw05, PAL["vw05"], "-",  r"VW(0.5)"),
    (rp_vwnz, PAL["vwnz"], "-",  r"VW(NZ)"),
]:
    cum = (1 + rp.dropna()).cumprod()
    ax.plot(cum.index, cum.values, color=color, ls=ls, lw=1.8, label=lab)
    label_endpoint(ax, cum.index[-1], cum.iloc[-1], f" {cum.iloc[-1]:.2f}×", color)
ax.set_title("Cumulative return (base = 1, Jan 2014)")
ax.set_ylabel("Wealth multiple")
ax.axhline(1, color="#BBBBBB", lw=0.6, zorder=0)
shade_events(ax, label_first=True)
ax.legend(loc="upper left")
year_axis(ax)

# ── Panel 2: CF + glide paths ─────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(vw_carbon.index,   vw_carbon["CF"],   "x--", color=PAL["vw"],   label="VW realized", lw=1.5)
ax.plot(vw05_carbon.index, vw05_carbon["CF"], "s-",  color=PAL["vw05"], label="VW(0.5) realized", lw=1.7)
ax.plot(vwnz_carbon.index, vwnz_carbon["CF"], "o-",  color=PAL["vwnz"], label="VW(NZ) realized", lw=1.7)
ax.plot(cf_target_nz.index, cf_target_nz.values, ":", color=PAL["vwnz"], lw=1.4, label="NZ target")
ax.plot(cf_target_vw.index, cf_target_vw.values, ":", color=PAL["vw05"], lw=1.4, label="0.5×VW target")
ax.set_title("CF evolution + glide paths")
ax.set_ylabel(r"tCO$_2$ / M\$ inv")
ax.legend(fontsize=8)

# ── Panel 3: WACI ─────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(vw_carbon.index,   vw_carbon["WACI"],   "x--", color=PAL["vw"],   label="VW", lw=1.5)
ax.plot(vw05_carbon.index, vw05_carbon["WACI"], "s-",  color=PAL["vw05"], label="VW(0.5)", lw=1.7)
ax.plot(vwnz_carbon.index, vwnz_carbon["WACI"], "o-",  color=PAL["vwnz"], label="VW(NZ)", lw=1.7)
ax.set_title("WACI evolution")
ax.set_ylabel(r"tCO$_2$ / M\$ rev")
ax.legend()

# ── Panel 4: cumulative active return ─────────────────────────────────────
ax = axes[1, 1]
ar05_cum = (1 + (rp_vw05 - rp_vw).dropna()).cumprod()
arnz_cum = (1 + (rp_vwnz - rp_vw).dropna()).cumprod()
ax.plot(ar05_cum.index, (ar05_cum - 1) * 100, color=PAL["vw05"], lw=1.6,
        label=f"VW(0.5) − VW (IR = {ir:+.2f})")
ax.plot(arnz_cum.index, (arnz_cum - 1) * 100, color=PAL["vwnz"], lw=1.6,
        label=f"VW(NZ) − VW (IR = {ir_nz:+.2f})")
ax.axhline(0, color="k", lw=0.6)
shade_events(ax, label_first=False)
ax.set_title("Cumulative active return vs VW")
ax.set_ylabel("Cumulative active return")
ax.yaxis.set_major_formatter(PCT0)
ax.legend(loc="lower left")
year_axis(ax)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_section41.{ext}")
plt.show()
print("   Saved: SAAM_Part2_section41.{pdf,png}")

---
### 25. §3.4 — Active vs passive: cost of decarbonization

| Metric | Active investor (MV → MV(0.5)) | Passive investor (VW → VW(0.5)) |
|---|---|---|
| Sharpe Δ | −0.001 | −0.025 |
| Volatility Δ | +0.04 pp | +0.17 pp |
| Geometric return Δ | −0.01 pp | −0.37 pp |
| Realized TE vs benchmark | n/a (vs VW: 13.4%) | 0.80% |
| IR (vs benchmark) | n/a | −0.41 |

**Mechanism.** The unconstrained MV portfolio was already aligned with low-carbon defensive sectors (banks, insurance, healthcare, staples). The CF constraint reinforces an existing tilt rather than imposing a new one — it's essentially free. The VW benchmark, by construction proportional to market cap, *cannot* be tilted away from the largest (often carbon-intensive) firms without taking active risk against itself. So the passive investor pays a measurable, but statistically insignificant, cost.

**Composition mechanics.** Both decarbonized portfolios target the same firms (Holcim, Heidelberg, RWE, ENEL, Maersk) but at very different scales. MV(0.5) makes few large tilts (largest 2013 cut: United Utilities at −4.62 pp); VW(0.5) makes many small tilts (largest 2013 cut: Holcim at −0.32 pp). Direct from the objectives — MV is unconstrained on benchmark proximity, while TE-min spreads its tilts to stay close to the index.

---
### 26. New chart — Ex-ante vs realized TE (the LW shrinkage caveat)

This is the most important methodological caveat in your report (Findings §9). For both VW(0.5) and VW(NZ), the SLSQP-optimal ex-ante TE — the value the optimizer thinks it is producing under the LW-shrunk Σ — is roughly **16× smaller than the realized OOS TE**.

**Why?** The LW shrinkage target is *constant correlation*: every off-diagonal correlation is replaced with a single average $\bar\rho$. That target has no notion of a "carbon-sector factor". The active position the optimizer takes (long staples / short utilities-energy-materials) lies along low-eigenvalue directions of the shrunk Σ, so the optimizer assigns near-zero forecast variance to that direction. But realized returns — especially the 2021–2022 energy shock — show the carbon factor has real volatility.

**Implication.** Ex-ante TE is fine as a *constraint-binding diagnostic* (the constraint binds in every year, the optimizer converges) but cannot be used as a forecast of realized risk. A factor risk model (Barra / Axioma / MSCI) with an explicit carbon factor would forecast TE accurately; we use ex-post statistics for performance attribution throughout.

In [ ]:
# Aggregate ex-ante TE (mean across years from solver logs) vs realized TE
ex_ante_05 = solver_df_vw["ann_TE_pct"].mean()
ex_ante_nz = solver_df_nz["ann_TE_pct"].mean()
real_05 = te_ep
real_nz = te_ep_nz

te_compare = pd.DataFrame({
    "Ex-ante TE (mean across years, %)": [ex_ante_05, ex_ante_nz],
    "Realized TE (full sample, %)":      [real_05,   real_nz],
}, index=["VW(0.5)", "VW(NZ)"])
te_compare["Ratio (realized / ex-ante)"] = te_compare["Realized TE (full sample, %)"] / te_compare["Ex-ante TE (mean across years, %)"]
display(te_compare.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: side-by-side bars ────────────────────────────────────────────
ax = axes[0]
labels = ["VW(0.5)", "VW(NZ)"]
xpos = np.arange(len(labels))
w = 0.35
b1 = ax.bar(xpos - w/2, [ex_ante_05, ex_ante_nz], w, color=PAL["vw"],
            label="Ex-ante TE (LW-shrunk Σ)", alpha=0.85)
b2 = ax.bar(xpos + w/2, [real_05,   real_nz],   w, color=PAL["high_e"],
            label="Realized TE (out-of-sample)", alpha=0.85)
for b, val in zip(b1, [ex_ante_05, ex_ante_nz]):
    ax.text(b.get_x() + b.get_width()/2, val + 0.02, f"{val:.2f}%", ha="center",
            fontsize=9, color=PAL["vw"], fontweight="bold")
for b, val in zip(b2, [real_05, real_nz]):
    ax.text(b.get_x() + b.get_width()/2, val + 0.02, f"{val:.2f}%", ha="center",
            fontsize=9, color=PAL["high_e"], fontweight="bold")
# Ratio annotation between bars
for i, (a, r) in enumerate(zip([ex_ante_05, ex_ante_nz], [real_05, real_nz])):
    ax.annotate(f"{r/a:.0f}×", xy=(i, max(a, r) + 0.18), ha="center",
                fontsize=11, fontweight="bold", color="#7A4017",
                bbox=dict(boxstyle="round,pad=0.3", fc="#FFF3E0", ec="#7A4017", lw=0.8))
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylabel("Annualized TE (%)")
ax.yaxis.set_major_formatter(PCT1)
ax.set_title("Ex-ante TE under LW-shrunk Σ vs realized TE (OOS)")
ax.legend(loc="upper left")
ax.set_ylim(0, max(real_05, real_nz) * 1.30)

# ── Panel 2: per-year ex-ante TE drift ─────────────────────────────────────
ax = axes[1]
ax.plot(solver_df_vw.index, solver_df_vw["ann_TE_pct"], "s-", color=PAL["vw05"],
        label="VW(0.5) ex-ante", lw=1.6)
ax.plot(solver_df_nz.index, solver_df_nz["ann_TE_pct"], "o-", color=PAL["vwnz"],
        label="VW(NZ) ex-ante", lw=1.6)
ax.axhline(real_05, color=PAL["vw05"], ls="--", lw=1.2,
           label=f"VW(0.5) realized ({real_05:.2f}%)")
ax.axhline(real_nz, color=PAL["vwnz"], ls="--", lw=1.2,
           label=f"VW(NZ) realized ({real_nz:.2f}%)")
ax.set_title("Per-year ex-ante TE vs realized (full-sample)")
ax.set_ylabel("Annualized TE (%)")
ax.set_xlabel("Year")
ax.yaxis.set_major_formatter(PCT1)
ax.legend(fontsize=8, loc="center right")

fig.suptitle("Ex-ante TE underestimates realized TE by ~16× — the LW-shrinkage caveat",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_TE_underestimation.{ext}")
plt.show()
print("   Saved: SAAM_Part2_TE_underestimation.{pdf,png}")

---
### 27. New chart — Concentration of carbon contribution (top-10 share over time)

Shows the trend explicitly. As the rest of the European market decarbonized organically by 58%, the top-10 hard-to-abate names took up a larger share of what remained: **43.5% (2013) → 45.2% (2018) → 51.5% (2024)**. By 2024 ten firms account for over half of the entire benchmark's WACI.

**Why this matters for the carbon constraint design.** Concentrated emissions mean concentrated tilts. The same five names (Holcim, RWE, Heidelberg, ENEL, Maersk) drive the active position year after year. A small number of "transition outcomes" at those firms move the entire cost-of-decarbonization story.

In [ ]:
# Compute top-10 contributor share for each year
def top10_share_of_waci(Y):
    isins = universe[Y]
    w = vw_weights(Y)
    CI = (co2_tot.loc[isins, Y] / rev_m.loc[isins, Y])
    contrib = (w * CI).dropna()
    if contrib.sum() <= 0:
        return np.nan
    contrib_sorted = contrib.sort_values(ascending=False)
    return contrib_sorted.head(10).sum() / contrib.sum() * 100

share_series = pd.Series({Y: top10_share_of_waci(Y) for Y in years_part2},
                         name="Top-10 share of VW WACI (%)")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(share_series.index, share_series.values, "o-", color=PAL["high_e"],
        lw=2.0, markersize=6)
ax.fill_between(share_series.index, share_series.values, 40, alpha=0.12, color=PAL["high_e"])

# Endpoint annotations
ax.annotate(f"{share_series.iloc[0]:.1f}%", xy=(share_series.index[0], share_series.iloc[0]),
            xytext=(-30, -15), textcoords="offset points",
            fontsize=10, color=PAL["high_e"], fontweight="bold")
ax.annotate(f"{share_series.iloc[-1]:.1f}%", xy=(share_series.index[-1], share_series.iloc[-1]),
            xytext=(8, 0), textcoords="offset points",
            fontsize=10, color=PAL["high_e"], fontweight="bold")

# Trend annotation
mid_y = years_part2[len(years_part2) // 2]
ax.text(mid_y, share_series.min() + 0.5,
        "As the rest of the market decarbonizes,\n"
        "the few hard-to-abate names take a larger share.",
        ha="center", fontsize=10, color="#7A4017", style="italic",
        bbox=dict(boxstyle="round,pad=0.5", fc="#FFF3E0", ec="#D9A066", lw=0.6))

ax.set_title("Top-10 firms' share of VW WACI over time (concentration of carbon contribution)")
ax.set_xlabel("Year")
ax.set_ylabel("Share of WACI accounted for by top-10 contributors")
ax.yaxis.set_major_formatter(PCT0)
ax.set_xticks(years_part2)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_top10_concentration.{ext}")
plt.show()
print("   Saved: SAAM_Part2_top10_concentration.{pdf,png}")

# Show the actual top-10 firms in 2024 alongside the chart
print("\n— Top-10 contributors to VW WACI in 2024 —")
display(top_n_by_contrib(2024, vw_weights(2024), 10, label="VW WACI"))

---
### 28. New chart — Country tilt of decarbonized portfolios

The findings doc describes the active position as a **"long staples / short utilities-energy-materials" sector tilt**. Without a sector field in `Static_2025.xlsx`, we use *country* as a proxy — it's a coarser slice but still tells a story, since utilities-energy-materials tend to be concentrated in specific geographies (UK / Germany / France / Switzerland for big industrials; Nordics for large-cap energy).

For each country we sum the active position $\sum_{i \in c} (\alpha_i^{(carbon)} - \alpha_i^{(vw)})$ in 2024 — top-10 absolute exposures.

In [ ]:
def country_tilt(w_port, w_bench, top_n=10):
    isin_to_country = static.set_index("ISIN")["Country"].to_dict()
    diff = (w_port - w_bench)
    rows = []
    for isin, dlt in diff.items():
        rows.append({"Country": isin_to_country.get(isin, "—"),
                     "Active_pp": dlt * 100})
    by_country = pd.DataFrame(rows).groupby("Country")["Active_pp"].sum()
    # Top-N by absolute magnitude
    top = by_country.reindex(by_country.abs().sort_values(ascending=False).head(top_n).index)
    return top.sort_values()

Y_focus = 2024
w_vw_2024  = vw_weights(Y_focus).reindex(universe[Y_focus]).fillna(0.0)
w_v05_2024 = vw05_w_dict[Y_focus].reindex(universe[Y_focus]).fillna(0.0)
w_vnz_2024 = vwnz_w_dict[Y_focus].reindex(universe[Y_focus]).fillna(0.0)

tilt_05 = country_tilt(w_v05_2024, w_vw_2024, top_n=10)
tilt_nz = country_tilt(w_vnz_2024, w_vw_2024, top_n=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)

for ax, tilt, title, color in [
    (axes[0], tilt_05, f"VW(0.5) − VW active position by country, {Y_focus}", PAL["vw05"]),
    (axes[1], tilt_nz, f"VW(NZ)  − VW active position by country, {Y_focus}", PAL["vwnz"]),
]:
    colors = [color if v >= 0 else PAL["high_e"] for v in tilt.values]
    bars = ax.barh(tilt.index, tilt.values, color=colors, alpha=0.85, edgecolor="white")
    for b, val in zip(bars, tilt.values):
        ax.text(b.get_width() + (0.005 if val >= 0 else -0.005),
                b.get_y() + b.get_height()/2,
                f"{val:+.2f}", ha="left" if val >= 0 else "right", va="center",
                fontsize=8.5, color="#444444")
    ax.axvline(0, color="k", lw=0.6)
    ax.set_title(title)
    ax.set_xlabel("Active position (pp)")
    ax.grid(axis="x")
    ax.set_axisbelow(True)

fig.suptitle("Country tilt of decarbonized passive portfolios vs VW  "
             "(green/teal = overweight, red = underweight)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_country_tilt.{ext}")
plt.show()
print("   Saved: SAAM_Part2_country_tilt.{pdf,png}")
print("\n   Note: country is a proxy for sector here. For a true sector decomposition,")
print("   add a sector / GICS field to Static_2025.xlsx.")

---
### 29. New chart — Glide path comparison: NZ target vs 0.5×VW vs VW realized

This is the chart that makes the "surprise" finding visible. For every year of our window, the orange-shaded zone shows that **the 0.5×VW target sits below (i.e. tighter than) the NZ glide path**. The NZ trajectory is a 10%/year decline anchored at 213.06 (2013); 0.5×VW is a moving target, halving whatever the benchmark realizes that year.

The benchmark itself decarbonized at an effective ~7%/year over 2013–2024. **A relative target inherits this; an absolute target does not.** Net-zero pledges anchored to absolute historical baselines silently weaken when the benchmark moves faster than the pledge.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))

# Three series
ax.plot(vw_carbon.index,   vw_carbon["CF"],   "x--", color=PAL["vw"],
        lw=1.8, label="VW realized CF (benchmark)", markersize=7)
ax.plot(cf_target_nz.index, cf_target_nz.values, "o-",  color=PAL["vwnz"],
        lw=2.0, label="NZ target (10%/yr from 2013)", markersize=6)
ax.plot(cf_target_vw.index, cf_target_vw.values, "s-",  color=PAL["vw05"],
        lw=2.0, label="0.5×VW target (relative)", markersize=5)

# Shading: where 0.5×VW is tighter (always in our window — illustrate the gap)
ax.fill_between(cf_target_nz.index,
                cf_target_nz.values, cf_target_vw.values,
                where=cf_target_vw.values < cf_target_nz.values,
                color=PAL["high_e"], alpha=0.14,
                label="0.5×VW tighter than NZ")

# Annotation: percentage difference at endpoint
end_year = years_part2[-1]
nz_end   = cf_target_nz.loc[end_year]
v05_end  = cf_target_vw.loc[end_year]
gap_pct  = (1 - v05_end / nz_end) * 100
ax.annotate(f"By {end_year}, 0.5×VW is\n{gap_pct:.0f}% tighter than NZ",
            xy=(end_year, (nz_end + v05_end) / 2),
            xytext=(-200, 30), textcoords="offset points",
            fontsize=10, color="#7A4017", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#7A4017", lw=0.9),
            bbox=dict(boxstyle="round,pad=0.4", fc="#FFF3E0", ec="#D9A066", lw=0.6))

# Highlight 2013 anchor
ax.scatter([2013], [CF_VW_2013], s=120, facecolors="none",
           edgecolors=PAL["vw"], lw=1.8, zorder=5)
ax.annotate(f"2013 anchor: {CF_VW_2013:.0f}",
            xy=(2013, CF_VW_2013), xytext=(10, 10), textcoords="offset points",
            fontsize=9, color=PAL["vw"])

ax.set_title("Glide path comparison — relative vs absolute carbon targets",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel(r"Carbon footprint (tCO$_2$ / M\$ inv)")
ax.set_xticks(years_part2)
ax.legend(loc="upper right")

# Secondary y-axis: % of 2013 baseline
def cf_to_pct(x): return x / CF_VW_2013 * 100
def pct_to_cf(x): return x * CF_VW_2013 / 100
ax2 = ax.secondary_yaxis("right", functions=(cf_to_pct, pct_to_cf))
ax2.set_ylabel("% of 2013 VW baseline")
ax2.yaxis.set_major_formatter(PCT0)

plt.tight_layout()
for ext in ("pdf", "png"):
    plt.savefig(f"{OUT}SAAM_Part2_glide_path_compare.{ext}")
plt.show()
print("   Saved: SAAM_Part2_glide_path_compare.{pdf,png}")

---
### 30. §4.2 — Net-zero discussion

The chosen $\theta = 10\%$/year corresponds to a Paris-aligned trajectory targeting near-zero emissions by approximately 2065 (since $0.9^{60} \approx 0.002$). This is more aggressive than the IPCC 1.5°C-aligned reduction rate of approximately 7%/year. A more aggressive investor ($\theta = 12\%$) would reach <1% of 2013 baseline by 2050; our results suggest the marginal cost of going from 10% to 12% would be modest in the European market.

**The deeper lesson** — illustrated explicitly in §29 — is that **the choice of baseline matters more than the steepness of the glide path** in periods of structural sector transition. An investor who pledged a 10%/year reduction from 2013 levels in 2013 would have nominally honoured their commitment in 2024, but they would have been doing *less* than the market index, which decarbonized at an effective ~7%/year on its own.

This is not hypothetical. Most net-zero asset-owner pledges (Net Zero Asset Owner Alliance, GFANZ) anchor on absolute baselines and have been criticized for exactly this asymmetry. Two cleaner framings:

1. **Carbon budget.** Constrain $\sum_Y \mathrm{CF}_Y$ over the investment horizon, not annual $\mathrm{CF}_Y$. Rewards early action and punishes delay regardless of benchmark trajectory.
2. **Relative-to-benchmark.** Constrain $\mathrm{CF}_Y \le (1-k) \cdot \mathrm{CF}(VW)_Y$. Forces continuous outperformance vs the market and avoids the structural-decarbonization free-rider effect.

---
### 31. Methodological caveats

**Ex-ante vs ex-post tracking error.** Diagnosed and visualised in §26. Mean ex-ante TE ≈ 0.05% annualized; realized ≈ 0.80%. The 16× ratio reflects that LW's constant-correlation target damps the carbon-sector factor. We use ex-post statistics for performance attribution and ex-ante TE only as a constraint-binding diagnostic.

**Carbon coverage.** WACI and CF are computed on the carbon-data subset, with weight coverage ≥ 99% in all years. The residual gap reflects firms reporting emissions but not revenue. This follows PCAF / MSCI methodology and avoids the renormalization bias that would arise from rescaling weights to the covered subset.

**Survivorship.** OOS returns set delisted firms to −100% in the delisting month and zero thereafter — the most punitive treatment, avoiding survivorship bias from dropping delisted firms from the universe.

**Carbon-portfolio inconsistency.** The Part I universe filter requires both Scope 1 and Scope 2 to be present, but does not require revenue or year-end Cap. With ≥ 99% weight coverage observed empirically, this gap was not material. A stricter filter would have improved coverage to 100% but would have departed from the project specification.

**Sample size.** 12 years OOS contains one major market crisis (COVID, Q1 2020) and one major sector regime shift (energy bull market, 2021–2022). Statistical inference about IRs and active returns is therefore limited; the −0.41 IR for VW(0.5) cannot be distinguished from zero at conventional levels.

**No explicit per-firm weight cap.** Long-only is the only individual-position constraint. As Jagannathan & Ma (2003) show, this acts as implicit shrinkage. A 5%–10% cap would be a useful robustness check.

**Sector data not in static file.** The "long staples / short utilities-energy-materials" tilt described in the findings doc is shown by country (§28) as a proxy. Adding a sector / GICS field to `Static_2025.xlsx` would unlock a true sector decomposition.

### 32. Conclusion

For a European-equity active investor minimizing variance with Scope 1+2 emissions data, **halving the portfolio's carbon footprint relative to its unconstrained baseline is essentially free** in our 2014–2025 sample (Sharpe drops by 0.001). The unconstrained MV portfolio is already aligned with low-carbon defensive sectors, so the carbon constraint reinforces an existing tilt.

For a passive investor minimizing tracking error against the value-weighted benchmark, **halving CF costs approximately 30 bp/year of return at 80 bp/year of TE (IR = −0.41)**. This cost is statistically indistinguishable from zero given the 12-year sample, and is concentrated in the 2021–2022 energy bull market.

A 10%/year net-zero glide path anchored at 2013 was **looser than a "CF ≤ 50% of benchmark" rule in every year of our window**. Structural sector decarbonization meant the relative target became progressively more demanding while the absolute glide path stayed fixed. **Net-zero pledges anchored to absolute historical baselines silently weaken when their benchmark decarbonizes faster than the pledged rate** — a generalizable observation with direct implications for how asset-owner net-zero commitments should be structured.

---
*All figures are saved to `Output_2026/` as both PDF and PNG. The official template (`SAAM_Part1_EUR_template.xlsx`) and extended results workbook (`SAAM_Part1_EUR_results.xlsx`) are also written to that directory.*